## limitata

In [1]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 20 SKU misto — VERSIONE CON I TRE FIX APPLICATI
================================================================================
Parte dalla versione "Benders Multi-Cut CORRETTO" (cella 2 del notebook
caricato), che gia' includeva i fix FIX_BIGM/FIX_MAP/FIX_INFEAS e la
struttura Base Stock (Acquisto+Immobilizzo). A quella base sono stati
aggiunti i TRE fix mancanti, verificati assenti in ENTRAMBE le celle del
notebook originale (sia la pipeline CG->MILP della cella 1, sia il modello
Benders-capable della cella 2):

  [FIX_MCCORMICK]  Base stock tecnologia-specifico. Prima: cost_BS usava
                    sempre c_CM[i], anche per le SKU AM. Ora: B_cm_lin
                    (gated da w[i,d]) usa c_CM, B_am_lin (gated da v[i,d])
                    usa prod_i — linearizzazione McCormick standard di
                    B[i,d]*w[i,d] e B[i,d]*v[i,d].

  [FIX_BACKORDER_LT] Costo di backorder analitico da lead time, assente
                    prima. bo_cm_aux/bo_am_aux via PWL su mu_var (stessa
                    logica di beta_cm_aux/beta_am_aux gia' presenti),
                    BO_act seleziona il valore giusto per tecnologia via
                    Big-M gated da cm[i]/am[i] e dc[d,i]. Aggiunto a
                    cost_bo_lt nell'obiettivo di PRIMO STADIO, quindi vale
                    automaticamente sia per il full-space sia per Benders
                    (theta[s] resta un costo di secondo stadio puro).

  [FIX_RAW_DOUBLE_COUNT] cost_raw = c_raw*mu_raw_contrib RIMOSSO
                    dall'obiettivo (prima presente, causava doppio
                    conteggio: prod_i = n_i*c_CM e' gia' "tutto incluso"
                    per Knofius et al. — il materiale e' gia' dentro il
                    costo unitario AM). La macchina EOQ_raw/capacita'
                    stampante (mu_raw_contrib, mu_raw_total, n3DP) resta
                    intatta: serve ancora per EOQ scorta polvere e per il
                    vincolo di capacita' di stampa, solo il termine di
                    COSTO diretto del materiale viene tolto.

  [FIX_VARHINT]     L'euristica di branching (VarHintVal/Pri) usava
                    c_CM anche per stimare il risparmio AM. Ora usa
                    prod_i (c_am) per il ramo AM, coerentemente con
                    cost_BS, e include il termine di backorder LT nella
                    stima di breakeven (stessa logica del breakeven a
                    fondo file).

NOTA: le due celle del notebook originale duplicavano lo stesso modello
in due implementazioni parallele (pipeline CG->MILP piu' semplice in
cella 1, versione Benders-capable in cella 2) — e nessuna delle due
aveva i tre fix. Questo file sostituisce ENTRAMBE: la Column Generation
qui sotto e' la stessa di prima ma con gli stessi tre fix applicati
internamente, cosi' il warm-start resta coerente con il MILP finale.
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time

# ==============================================================================
# PARAMETRI GLOBALI - 20 SKU (identici al notebook originale)
# ==============================================================================
nI, nD, nC = 20, 3, 6
seed = 42

coord_DC = np.array([[1.0, 8.0], [1.0, 2.0], [5.0, 5.0]])
coord_C = np.array([[0.5, 9.0], [2.0, 8.0], [3.5, 5.5],
                     [5.5, 4.5], [1.5, 2.5], [0.5, 1.0]])

categorie = (["strutt."] * 2 + ["LT-driver"] * 5 +
             ["borderline"] * 5 + ["meccanico"] * 4 + ["consumab."] * 4)

c_CM = np.array([4000., 3500., 5000., 4000., 3000., 2000., 1500.,
                  1200., 900., 700., 500., 300., 200., 150., 100., 80.,
                  40., 25., 15., 10.])
ratio = np.array([0.38, 0.42, 1.20, 1.30, 1.40, 1.50, 1.60,
                   2.00, 2.50, 3.00, 3.00, 3.50, 3.50, 4.00, 4.50, 5.00,
                   4.00, 5.00, 5.50, 6.00])
c_AM = c_CM * ratio
n_i = ratio

L_CM = np.array([35, 30, 40, 35, 30, 28, 25, 20, 18, 16, 14, 12, 12, 10, 8, 6, 8, 6, 5, 3]) / 52
L_AM_vec = np.array([1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 4]) / 52
L_AM = np.column_stack([L_AM_vec] * nD)

b_i = np.array([100000., 80000., 150000., 120000., 80000., 50000., 40000.,
                15000., 12000., 10000., 8000., 5000., 3000., 2000., 1500., 1000.,
                600., 400., 250., 150.])
SL_i = np.array([0.99, 0.99, 0.99, 0.99, 0.98, 0.97, 0.97,
                  0.95, 0.95, 0.93, 0.92, 0.90, 0.90, 0.90, 0.88, 0.88,
                  0.85, 0.83, 0.82, 0.80])

mu_ic = np.array([
    [0.6, 0.5, 0.4, 0.3, 0.3, 0.3], [0.5, 0.4, 0.4, 0.3, 0.2, 0.2],
    [0.5, 0.4, 0.4, 0.3, 0.4, 0.3], [0.6, 0.4, 0.4, 0.3, 0.4, 0.3],
    [0.8, 0.6, 0.5, 0.4, 0.5, 0.4], [1.0, 0.7, 0.7, 0.6, 0.6, 0.6],
    [1.1, 0.8, 0.7, 0.6, 0.7, 0.7], [1.2, 1.0, 0.9, 0.8, 0.9, 0.8],
    [1.3, 1.1, 1.0, 0.8, 1.0, 0.9], [1.4, 1.1, 1.0, 0.9, 1.0, 0.9],
    [1.0, 0.8, 0.8, 0.7, 0.8, 0.7], [0.9, 0.7, 0.7, 0.6, 0.7, 0.6],
    [2.5, 2.0, 1.8, 1.5, 1.5, 1.2], [2.0, 1.8, 1.5, 1.2, 1.2, 1.0],
    [1.8, 1.5, 1.2, 1.0, 1.0, 0.8], [1.5, 1.2, 1.0, 0.8, 0.8, 0.7],
    [4.0, 3.5, 3.0, 2.5, 2.5, 2.0], [3.5, 3.0, 2.5, 2.0, 2.0, 1.5],
    [3.0, 2.5, 2.0, 1.8, 1.5, 1.5], [2.5, 2.0, 1.8, 1.5, 1.2, 1.2],
])

alpha_ic = np.column_stack([SL_i] * nC)
a_i = c_AM / (1.30 * 0.00525 * 3600)
h_d = np.full(nD, 0.25)
Leas = 15000.

dist = np.array([[np.sqrt((coord_DC[d, 0] - coord_C[c, 0]) ** 2 +
                           (coord_DC[d, 1] - coord_C[c, 1]) ** 2)
                   for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist * 80.0] * nI, axis=1)


# ==============================================================================
# FUNZIONI ANALITICHE (identiche all'originale)
# ==============================================================================
def expected_backorder(mu_L, beta):
    if mu_L < 1e-9 or beta < 0: return mu_L
    if beta == 0: return mu_L
    return max(mu_L * (1 - poisson.cdf(beta - 1, mu_L)) - beta * (1 - poisson.cdf(beta, mu_L)), 0.0)


def expected_on_hand(mu_L, beta):
    return max(beta - mu_L + expected_backorder(mu_L, beta), 0.0)


def optimal_beta(mu_L, c_hold, b_val):
    if mu_L < 1e-9: return 0, 0.0
    best_cost, best_b = float('inf'), 0
    for b in range(0, int(mu_L * 4) + 20):
        tot = c_hold * b + b_val * expected_backorder(mu_L, b)
        if tot < best_cost:
            best_cost, best_b = tot, b
        elif tot > best_cost * 1.5 and b > mu_L:
            break
    return best_b, best_cost


def beta_poisson(mu_val, L_val, SL_target):
    lam = mu_val * L_val
    if lam <= 1e-9: return 0
    for s in range(100000):
        if poisson.cdf(s, lam) >= SL_target: return int(s)
    return 99999


# ==============================================================================
# COLUMN GENERATION — CON I TRE FIX APPLICATI
# ==============================================================================
def precompute_all_columns(nI, nD, nC, p):
    """
    [FIX_MCCORMICK] branch AM: c_hold ora usa c_am (= p['c_CM'][i]*p['n_i'][i]),
                    non piu' p['c_CM'][i].
    [FIX_BACKORDER_LT] aggiunto bo_lt = p['b_i'][i]*expected_backorder(mu_L, beta)
                    a entrambi i branch (CM e AM), sommato al costo colonna.
    [FIX_RAW_DOUBLE_COUNT] rimosso il termine 'raw = c_raw*qraw*mu_id' dal
                    branch AM — prod = c_am*mu_id e' gia' tutto incluso.
    """
    t0 = time.time()
    table = {}
    n_entries = 0
    for i in range(nI):
        table[i] = {}
        for d in range(nD):
            table[i][d] = {}
            for tech in ['CM', 'AM']:
                table[i][d][tech] = {}
                for mask in range(1, 2 ** nC):
                    clients = [c for c in range(nC) if mask & (1 << c)]
                    mu_id = sum(p['mu_ic'][i, c] for c in clients)
                    if mu_id < 1e-9: continue
                    trans = sum(p['t_dic'][d, i, c] * p['mu_ic'][i, c] for c in clients)

                    if tech == 'CM':
                        mu_L = mu_id * p['L_CM'][i]
                        c_hold = p['c_CM'][i] * (1 + p['h_d'][d])
                        beta, bs_back = optimal_beta(mu_L, c_hold, p['b_i'][i])
                        eoq = np.sqrt(2 * p['o'] * p['h_d'][d] * p['c_CM'][i] * mu_id)
                        prod = p['c_CM'][i] * mu_id
                        hold_res = p['h_d'][d] * p['c_CM'][i] * expected_on_hand(mu_L, beta)
                        bo_lt = p['b_i'][i] * expected_backorder(mu_L, beta)   # [FIX_BACKORDER_LT]
                        cost = bs_back + eoq + prod + trans + hold_res + bo_lt
                        ph = 0
                    else:
                        L_am = p['L_AM'][:, 0][i]
                        mu_L = mu_id * L_am
                        c_am = p['c_CM'][i] * p['n_i'][i]
                        c_hold = c_am * (1 + p['h_d'][d])                     # [FIX_MCCORMICK]
                        beta, bs_back = optimal_beta(mu_L, c_hold, p['b_i'][i])
                        prod = c_am * mu_id
                        vol = c_am / (1.30 * 1e6)
                        qraw = vol * p['den_raw'] * p['unit_raw']
                        # [FIX_RAW_DOUBLE_COUNT] rimosso: raw = p['c_raw']*qraw*mu_id
                        eoq_raw = np.sqrt(2 * p['o_raw'] * p['h_d'][d] * p['c_raw'] * max(qraw * mu_id, 1e-9))
                        hold_raw = p['h_d'][d] * p['c_raw'] * qraw * mu_id * 0.5
                        hold_res = p['h_d'][d] * c_am * expected_on_hand(mu_L, beta)  # [FIX_MCCORMICK] c_am non c_CM
                        bo_lt = p['b_i'][i] * expected_backorder(mu_L, beta)   # [FIX_BACKORDER_LT]
                        a_i_local = c_am / (1.30 * 0.00525 * 3600)
                        ph = a_i_local * mu_id
                        cost = bs_back + prod + eoq_raw + hold_raw + trans + hold_res + bo_lt

                    table[i][d][tech][mask] = (cost, mu_id, beta, ph)
                    n_entries += 1

    elapsed = time.time() - t0
    print(f"  [CG] Precomputazione: {n_entries:,} entry in {elapsed:.0f}s")
    return table


def pricing_precomputed(duals_ic, w_vals, table, nI, nD, nC, Leas, existing):
    new_cols = []
    best_rc = 0
    for i in range(nI):
        for d in range(nD):
            for tech in ['CM', 'AM']:
                best_mask_rc, best_mask, best_entry = -1e12, 0, None
                for mask, entry in table[i][d][tech].items():
                    cost, mu, beta, ph = entry
                    dual_sum = sum(duals_ic[i, c] for c in range(nC) if mask & (1 << c))
                    rc = dual_sum - cost
                    if rc > best_mask_rc:
                        best_mask_rc, best_mask, best_entry = rc, mask, entry

                if best_entry is None: continue
                rc_adj = best_mask_rc
                if tech == 'AM' and w_vals[d] < 0.5:
                    rc_adj -= Leas * (1 - w_vals[d])

                if rc_adj > 1e-4:
                    key = (d, i, tech, best_mask)
                    if key not in existing:
                        cost, mu, beta, ph = best_entry
                        new_cols.append((d, i, tech, best_mask, cost, mu, beta, ph, rc_adj))
                        existing.add(key)
                        best_rc = max(best_rc, rc_adj)
    return new_cols, best_rc


def column_generation(params, nI, nD, nC, Leas, Cap_3DP=5000,
                       stock_capacity=None, max_iter=80, verbose=True):
    t0 = time.time()
    table = precompute_all_columns(nI, nD, nC, params)

    columns = []
    full_mask = (1 << nC) - 1
    for i in range(nI):
        for d in range(nD):
            for tech in ['CM', 'AM']:
                if full_mask in table[i][d][tech]:
                    cost, mu, beta, ph = table[i][d][tech][full_mask]
                    columns.append((d, i, tech, full_mask, cost, mu, beta, ph))
    existing = set((c[0], c[1], c[2], c[3]) for c in columns)

    mdl = gp.Model("CG")
    mdl.Params.OutputFlag = 0
    mdl.Params.Method = 1

    n3dp = {}
    for d in range(nD):
        n3dp[d] = mdl.addVar(lb=0, name=f"n3dp_{d}")
    cover = {}
    for i in range(nI):
        for c in range(nC):
            cover[i, c] = mdl.addConstr(gp.LinExpr() == 1, name=f"cv_{i}_{c}")
    cap_c = {}
    for d in range(nD):
        cap_c[d] = mdl.addConstr(-Cap_3DP * n3dp[d] <= 0, name=f"cap_{d}")

    mdl.setObjective(gp.quicksum(Leas * n3dp[d] for d in range(nD)), GRB.MINIMIZE)
    mdl.update()

    lam_vars = []
    for r, col in enumerate(columns):
        dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r = col
        gc = gp.Column()
        for c in range(nC):
            if mask_r & (1 << c): gc.addTerms(1.0, cover[sku_r, c])
        if tech_r == 'AM' and ph_r > 0: gc.addTerms(ph_r, cap_c[dc_r])
        lam_vars.append(mdl.addVar(lb=0, ub=1, obj=cost_r, column=gc, name=f"l_{r}"))
    mdl.update()

    for it in range(max_iter):
        mdl.optimize()
        if mdl.Status != 2: break
        lb = mdl.ObjVal
        duals_ic = np.zeros((nI, nC))
        for i in range(nI):
            for c in range(nC): duals_ic[i, c] = cover[i, c].Pi
        w_vals = np.array([n3dp[d].X for d in range(nD)])

        new_cols, best_rc = pricing_precomputed(duals_ic, w_vals, table, nI, nD, nC, Leas, existing)
        n_new = len(new_cols)

        for nc in new_cols:
            dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r, _ = nc
            columns.append((dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r))
            gc = gp.Column()
            for c in range(nC):
                if mask_r & (1 << c): gc.addTerms(1.0, cover[sku_r, c])
            if tech_r == 'AM' and ph_r > 0: gc.addTerms(ph_r, cap_c[dc_r])
            lam_vars.append(mdl.addVar(lb=0, ub=1, obj=cost_r, column=gc, name=f"l_{len(lam_vars)}"))
        mdl.update()

        if verbose and (it % 5 == 0 or n_new == 0):
            el = time.time() - t0
            print(f"  [CG] It {it:>3}: LB={lb:>10,.0f}  col={len(columns):>5}  "
                  f"new={n_new:>3}  rc={best_rc:>8,.0f}  t={el:.0f}s")
        if n_new == 0: break
    mdl.dispose()

    mdl2 = gp.Model("MILP"); mdl2.Params.OutputFlag = 0; mdl2.Params.TimeLimit = 120
    lam2 = mdl2.addVars(len(columns), vtype=GRB.BINARY, name="l")
    n3dp2 = mdl2.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n")
    for i in range(nI):
        for c in range(nC):
            mdl2.addConstr(gp.quicksum(lam2[r] for r in range(len(columns))
                                        if columns[r][1] == i and (columns[r][3] & (1 << c))) == 1)
    for d in range(nD):
        mdl2.addConstr(gp.quicksum(columns[r][7] * lam2[r] for r in range(len(columns))
                                    if columns[r][0] == d and columns[r][2] == 'AM' and columns[r][7] > 0)
                       <= Cap_3DP * n3dp2[d])
    mdl2.setObjective(gp.quicksum(columns[r][4] * lam2[r] for r in range(len(columns)))
                       + gp.quicksum(Leas * n3dp2[d] for d in range(nD)), GRB.MINIMIZE)
    mdl2.optimize()

    selected, obj_milp, gap, n3dp_sol = [], float('inf'), 100, [0] * nD
    if mdl2.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        selected = [r for r in range(len(columns)) if lam2[r].X > 0.5]
        obj_milp = mdl2.ObjVal; gap = mdl2.MIPGap * 100
        n3dp_sol = [int(round(n3dp2[d].X)) for d in range(nD)]
    mdl2.dispose()

    am = np.zeros(nI, dtype=int); dc_sol = np.zeros((nD, nI), dtype=int)
    y_sol = np.zeros((nD, nI, nC), dtype=int); B_sol = np.zeros((nI, nD))
    for r in selected:
        dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r = columns[r]
        if tech_r == 'AM': am[sku_r] = 1
        dc_sol[dc_r, sku_r] = 1; B_sol[sku_r, dc_r] = beta_r
        for c in range(nC):
            if mask_r & (1 << c): y_sol[dc_r, sku_r, c] = 1

    return {'am': am, 'dc': dc_sol, 'y': y_sol, 'B': B_sol, 'n3dp': n3dp_sol,
            'obj_lp': lb, 'obj_milp': obj_milp, 'gap': gap, 'time': time.time() - t0}


# ==============================================================================
# MODELLO PRIMO STADIO + SECONDO STADIO (full-space o Benders) — CON I FIX
# ==============================================================================
def solve_stochastic(
        nI, nD, nC, nS,
        mu_ic, alpha_ic,
        SL, SL_raw,
        c_CM, o, h_d, b_i, L_CM,
        n_i, c_raw, den_raw, unit_raw,
        o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i,
        t_central,
        B_cap=None,
        t_dic=None,
        costo_km=100.0,
        coord_DC=None,
        coord_C=None,
        L_card=None, U_card=None,
        use_benders=True,
        mip_gap=0.01, time_limit=600,
        xi=2.0, seed=42,
        warm_start_sol=None,
        fix_am=None,
        fix_cm=None,
        fix_n_dc=None,
        verbose=False, scenario_name="default"
):
    """
    RETURNS dict:
      status, obj, obj_real, gap, time,
      am, cm, dc, B, n3DP, n_dc_per_sku,
      y_sol, cost_breakdown, cuts_added
    """
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([
            [np.sqrt((coord_DC[d, 0] - coord_C[c, 0]) ** 2 +
                     (coord_DC[d, 1] - coord_C[c, 1]) ** 2)
             for c in range(nC)] for d in range(nD)])
        t_dc = dist * costo_km
        t_dic = np.stack([t_dc] * nI, axis=1)

    # ── 1. PREPROCESSING ─────────────────────────────────────────────────────
    prod_i = n_i * c_CM
    vol_i = (prod_i / 1.30) * 1e-6
    qbar_i = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    mu_grid_set = {0.0}
    for i in I:
        domande_i = [float(mu_ic[i, c]) for c in C]
        for r in range(1, nC + 1):
            for combo in combinations(domande_i, r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid = np.zeros((nI, nD, M))
    B_AM_grid = np.zeros((nI, nD, M))
    C_EOQ_grid = np.zeros((nI, nD, M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i, d, m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i, d, m] = beta_poisson(mu_m, L_AM[i, d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i, d, m] = np.sqrt(2 * o * h_d[d] * c_CM[i] * mu_m)

    B_bar = B_CM_grid[:, :, -1]
    mu_id_tot = mu_ic.sum(axis=1)

    contributi_raw = [float(qbar_i[i] * mu_id_tot[i]) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI + 1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        max_raw = sum(contributi_raw)
        mu_raw_grid = np.linspace(0, max_raw, 200)
    M_RAW = len(mu_raw_grid)

    B_raw_grid = np.zeros((nD, M_RAW))
    C_EOQ_raw_grid = np.zeros((nD, M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d, m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0:
                C_EOQ_raw_grid[d, m] = np.sqrt(2 * o_raw * h_d[d] * c_raw * mu_m)
        B_bar_raw[d] = B_raw_grid[d, -1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:, :, np.newaxis], size=(nI, nC, nS)).astype(float)

    Q_bar_CM = np.zeros((nI, nD))
    F_bar = np.zeros((nI, nD))
    for i in I:
        for d in D:
            Q_bar_CM[i, d] = d_ics[i, :, :].sum(axis=0).max() * xi
            F_bar[i, d] = d_ics[i, :, :].max() * xi
    Q_bar_AM = np.full((nI, nD), Cap_3DP * xi)

    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── 2. MODELLO GUROBI — PRIMO STADIO ─────────────────────────────────────
    mdl = gp.Model(f"stoc_benders_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    am = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")

    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")

    # [FIX_BACKORDER_LT] nuove variabili ausiliarie
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")

    if not use_benders:
        q_CM = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
        q_AM = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
        f = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
        u = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
        I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")
    else:
        theta = mdl.addVars(nS, lb=0.0, name="theta")

    # ── Vincoli primo stadio (identici all'originale) ────────────────────────
    mdl.addConstrs((am[i] + cm[i] == 1 for i in I), name="A1")
    mdl.addConstrs((y[d, i, c] <= dc[d, i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d, i] <= gp.quicksum(y[d, i, c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d, i, c] for d in D) == 1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d, i] for d in D) >= L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d, i] for d in D) <= U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i, d] <= am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i, d] <= dc[d, i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i, d] >= am[i] + dc[d, i] - 1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i, d] <= cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i, d] <= dc[d, i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i, d] >= cm[i] + dc[d, i] - 1 for i in I for d in D), name="W3")

    mu_max_i = np.array([mu_ic[i, :].sum() for i in I])
    mdl.addConstrs((mu_var[d, i] == gp.quicksum(mu_ic[i, c] * y[d, i, c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d, i] <= mu_max_i[i] * dc[d, i]
                    for d in D for i in I), name="DG2_dc")

    N_PWL = 30
    for i in I:
        mu_pts_i = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL).tolist()
        for d in D:
            beta_cm_pts = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in mu_pts_i]
            mdl.addGenConstrPWL(mu_var[d, i], beta_cm_aux[i, d], mu_pts_i, beta_cm_pts,
                                 name=f"PWL_bCM_{d}_{i}")
            beta_am_pts = [float(beta_poisson(mu, L_AM[i, d], SL_i[i])) for mu in mu_pts_i]
            mdl.addGenConstrPWL(mu_var[d, i], beta_am_aux[i, d], mu_pts_i, beta_am_pts,
                                 name=f"PWL_bAM_{d}_{i}")
            eoq_pts_id = [float(np.sqrt(2 * o * h_d[d] * c_CM[i] * mu)) if mu > 0 else 0.0 for mu in mu_pts_i]
            mdl.addGenConstrPWL(mu_var[d, i], C_EOQ_var[d, i], mu_pts_i, eoq_pts_id,
                                 name=f"PWL_EOQ_{d}_{i}")

            # [FIX_BACKORDER_LT] PWL del costo di backorder analitico da lead time,
            # calcolato punto-per-punto sulla stessa griglia mu_pts_i, usando il
            # beta gia' calcolato per quel punto (beta_cm_pts/beta_am_pts sopra).
            bo_cm_pts = [b_i[i] * expected_backorder(mu_pts_i[k] * L_CM[i], beta_cm_pts[k])
                         for k in range(len(mu_pts_i))]
            mdl.addGenConstrPWL(mu_var[d, i], bo_cm_aux[i, d], mu_pts_i, bo_cm_pts,
                                 name=f"PWL_boCM_{d}_{i}")
            bo_am_pts = [b_i[i] * expected_backorder(mu_pts_i[k] * L_AM[i, d], beta_am_pts[k])
                         for k in range(len(mu_pts_i))]
            mdl.addGenConstrPWL(mu_var[d, i], bo_am_aux[i, d], mu_pts_i, bo_am_pts,
                                 name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i, d] >= beta_cm_aux[i, d] - B_bar[i, d] * (1 - cm[i])
                    for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i, d] >= beta_am_aux[i, d] - B_bar[i, d] * (1 - am[i])
                    for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i, d] <= B_bar[i, d] * dc[d, i]
                    for i in I for d in D), name="DG5")

    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i, d] for i in I) <= B_cap
                        for d in D), name="DG5b")

    # [FIX_BACKORDER_LT] BO_act = valore del ramo tecnologico attivo (Big-M gated
    # da cm[i]/am[i], come gia' si fa per B in DG4a/DG4b), azzerato se dc[d,i]=0.
    M_BO_big = max(max(b_i[i] * mu_max_i[i] for i in I), 1.0) * 2
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d, i] >= bo_cm_aux[i, d] - M_BO_big * (1 - cm[i]), name=f"BOa_{d}_{i}")
            mdl.addConstr(BO_act[d, i] >= bo_am_aux[i, d] - M_BO_big * (1 - am[i]), name=f"BOb_{d}_{i}")
            mdl.addConstr(BO_act[d, i] <= bo_cm_aux[i, d] + M_BO_big * (1 - cm[i]), name=f"BOc_{d}_{i}")
            mdl.addConstr(BO_act[d, i] <= bo_am_aux[i, d] + M_BO_big * (1 - am[i]), name=f"BOd_{d}_{i}")
            mdl.addConstr(BO_act[d, i] <= M_BO_big * dc[d, i], name=f"BOe_{d}_{i}")

    for d in D:
        for i in I:
            mdl.addConstr(dcAM[d] >= v[i, d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d] <= gp.quicksum(v[i, d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i) * sum(mu_ic[i, :].sum() for i in I)) * 2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d, i] <= M_raw_big * v[i, d], name=f"RC1_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d, i] <= qbar_i[i] * mu_var[d, i], name=f"RC2_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d, i] >= qbar_i[i] * mu_var[d, i] - M_raw_big * (1 - v[i, d]),
                          name=f"RC3_{d}_{i}")
        mdl.addConstr(mu_raw_total[d] == gp.quicksum(mu_raw_contrib[d, i] for i in I),
                      name=f"mu_raw_tot_{d}")
        mdl.addConstr(mu_raw_total[d] <= M_raw_big * dcAM[d],
                      name=f"mu_raw_dc_{d}")

    N_PWL_RAW = 30
    mu_raw_max = float(sum(qbar_i[i] * mu_ic[i, :].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max, 0.01), N_PWL_RAW).tolist()
    for d in D:
        b_raw_pts = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        eoq_raw_pts = [float(np.sqrt(2 * o_raw * h_d[d] * c_raw * mu)) if mu > 0 else 0.0 for mu in mu_raw_pts]
        b_raw_aux = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], b_raw_aux, mu_raw_pts, b_raw_pts, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d] >= b_raw_aux, name=f"ER4_{d}")
        mdl.addConstr(B_raw[d] <= B_bar_raw[d] * dcAM[d], name=f"ER5_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, eoq_raw_pts, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max()) * 2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d, i] <= M_EOQ_big * w[i, d], name=f"L_EOQ1_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d, i] <= C_EOQ_var[d, i], name=f"L_EOQ2_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d, i] >= C_EOQ_var[d, i] - M_EOQ_big * (1 - w[i, d]), name=f"L_EOQ3_{d}_{i}")

    ratio_pt = np.where(qbar_i > 1e-9, ptime_i / qbar_i, 0.0)
    mdl.addConstrs((n3DP[d] >= gp.quicksum(ratio_pt[i] * mu_raw_contrib[d, i] for i in I) / Cap_3DP for d in D),
                    name="E1")

    # [FIX_MCCORMICK] B_cm_lin/B_am_lin — linearizzazione McCormick standard
    # di B[i,d]*w[i,d] e B[i,d]*v[i,d]. Sostituisce l'uso diretto di B[i,d]
    # con c_CM in cost_BS, che ignorava la tecnologia.
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i, d])
            mdl.addConstr(B_cm_lin[i, d] <= Bub * w[i, d], name=f"BCL1_{i}_{d}")
            mdl.addConstr(B_cm_lin[i, d] <= B[i, d], name=f"BCL2_{i}_{d}")
            mdl.addConstr(B_cm_lin[i, d] >= B[i, d] - Bub * (1 - w[i, d]), name=f"BCL3_{i}_{d}")
            mdl.addConstr(B_am_lin[i, d] <= Bub * v[i, d], name=f"BAL1_{i}_{d}")
            mdl.addConstr(B_am_lin[i, d] <= B[i, d], name=f"BAL2_{i}_{d}")
            mdl.addConstr(B_am_lin[i, d] >= B[i, d] - Bub * (1 - v[i, d]), name=f"BAL3_{i}_{d}")

    # [FIX_MCCORMICK] cost_BS ora tecnologia-specifico: c_CM per CM, prod_i per AM
    cost_BS = gp.quicksum(
        c_CM[i] * (1 + h_d[d]) * B_cm_lin[i, d] + prod_i[i] * (1 + h_d[d]) * B_am_lin[i, d]
        for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d, i] for i in I for d in D)
    # [FIX_RAW_DOUBLE_COUNT] cost_raw RIMOSSO (prima: c_raw*mu_raw_contrib, doppio
    # conteggio perche' prod_i include gia' il materiale "tutto incluso")
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d] + h_d[d] * c_raw * B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas * n3DP[d] for d in D)
    # [FIX_BACKORDER_LT] nuovo componente di costo di primo stadio
    cost_bo_lt = gp.quicksum(BO_act[d, i] for i in I for d in D)

    # ── 3. SECONDO STADIO: full-space o Benders ──────────────────────────────
    if not use_benders:
        mdl.addConstrs((gp.quicksum(a_i[i] * q_AM[i, d, s] for i in I) <= Cap_3DP * n3DP[d] for d in D for s in S),
                        name="E2")
        mdl.addConstrs((B[i, d] + q_CM[i, d, s] + q_AM[i, d, s] - gp.quicksum(f[i, d, c, s] for c in C) ==
                        I_inv[i, d, s] for i in I for d in D for s in S), name="F1")
        mdl.addConstrs(
            (gp.quicksum(f[i, d, c, s] for d in D) + u[i, c, s] == d_ics[i, c, s] for i in I for c in C for s in S),
            name="F2")
        mdl.addConstrs((f[i, d, c, s] <= F_bar[i, d] * dc[d, i] for i in I for d in D for c in C for s in S),
                        name="G1")
        mdl.addConstrs((q_CM[i, d, s] <= Q_bar_CM[i, d] * w[i, d] for i in I for d in D for s in S), name="G2")
        mdl.addConstrs((q_AM[i, d, s] <= Q_bar_AM[i, d] * v[i, d] for i in I for d in D for s in S), name="G3")
        mdl.addConstrs((gp.quicksum(pi_s * gp.quicksum(f[i, d, c, s] for d in D) for s in S) >=
                        alpha_ic[i, c] * gp.quicksum(pi_s * d_ics[i, c, s] for s in S) for i in I for c in C),
                        name="H")

        cost_2nd = gp.quicksum(pi_s * (
                gp.quicksum(c_CM[i] * q_CM[i, d, s] for i in I for d in D) +
                gp.quicksum(prod_i[i] * q_AM[i, d, s] for i in I for d in D) +
                gp.quicksum(t_dic[d, i, c] * f[i, d, c, s] for i in I for d in D for c in C) +
                gp.quicksum(h_d[d] * c_CM[i] * I_inv[i, d, s] for i in I for d in D) +
                gp.quicksum(b_i[i] * u[i, c, s] for i in I for c in C)
        ) for s in S)

        # [FIX_RAW_DOUBLE_COUNT] cost_raw tolto dalla somma; [FIX_BACKORDER_LT] cost_bo_lt aggiunto
        mdl.setObjective(cost_BS + cost_EOQ + cost_raw_dc + cost_leas + cost_bo_lt + cost_2nd, GRB.MINIMIZE)

        if fix_am is not None and fix_cm is not None:
            for i in I:
                am[i].lb = am[i].ub = float(bool(fix_am[i]))
                cm[i].lb = cm[i].ub = float(bool(fix_cm[i]))
            mdl.update()
            if verbose:
                n_fa = sum(bool(fix_am[i]) for i in I)
                print(f"  [fix_tech] {n_fa}/{nI} SKU fissati AM, {nI - n_fa}/{nI} CM.")

        # [FIX_VARHINT] hint di branching ora tecnologia-specifico: usa prod_i
        # (c_am) invece di c_CM per stimare il risparmio del ramo AM, e include
        # il termine di backorder LT nella stima di netto (coerente con cost_BS
        # e cost_bo_lt sopra, invece della vecchia stima solo-holding).
        for i in I:
            mu_c = float(mu_ic[i, :].sum())
            b_cm_h = beta_poisson(mu_c, L_CM[i], SL_i[i])
            b_am_h = beta_poisson(mu_c, L_AM[i, 0], SL_i[i])
            risp_hold = c_CM[i] * (1 + h_d[0]) * b_cm_h - prod_i[i] * (1 + h_d[0]) * b_am_h
            bo_cm_h = b_i[i] * expected_backorder(mu_c * L_CM[i], b_cm_h)
            bo_am_h = b_i[i] * expected_backorder(mu_c * L_AM[i, 0], b_am_h)
            risp = risp_hold + (bo_cm_h - bo_am_h)
            costo = (prod_i[i] - c_CM[i]) * mu_c
            netto = risp - costo
            Leas_hint = Leas if isinstance(Leas, float) else float(Leas)
            if netto > 0:
                am[i].VarHintVal = 1.0; am[i].VarHintPri = 10; cm[i].VarHintVal = 0.0
            elif -Leas_hint < netto <= 0:
                am[i].VarHintVal = 1.0; am[i].VarHintPri = 5; cm[i].VarHintVal = 0.0
            else:
                am[i].VarHintVal = 0.0; am[i].VarHintPri = 1; cm[i].VarHintVal = 1.0
        mdl.update()
        if verbose:
            n_hint_am = sum(1 for i in I if am[i].VarHintVal > 0.5)
            print(f"  [varhint] {n_hint_am}/{nI} SKU con hint AM.")

        # [FIX_WARMSTART_INJECTION] Iniezione del warm-start CG nel ramo
        # full-space -- corretta secondo i 3 punti gia' descritti in
        # main.md (subsec:meth_cg):
        #   1. NON iniettare B[i,d] -- il beta calcolato analiticamente
        #      dalla CG puo' differire di appena 1 unita' da quello che
        #      il motore PWL di Gurobi produce allo stesso mu_id, e
        #      questo basta a violare F1 e far scartare l'INTERO warm
        #      start. Le variabili continue restano a Gurobi.
        #   2. dcAM[d].Start aggiunto (mancava) -- 1 se almeno una SKU AM
        #      della CG e' assegnata a quel DC nel warm start.
        #   3. n3DP[d].Start ricalcolato sul PICCO di domanda tra gli
        #      scenari SAA (d_ics, gia' generato in fase di
        #      preprocessing con lo stesso seed/nS del modello finale),
        #      non sulla domanda media implicita nel vincolo di
        #      capacita' della CG -- altrimenti il punto iniettato puo'
        #      violare E2 in uno scenario di picco e farsi scartare.
        if warm_start_sol is not None:
            try:
                ws_am = warm_start_sol.get('am', np.zeros(nI, dtype=bool))
                ws_cm = warm_start_sol.get('cm', np.ones(nI, dtype=bool))
                ws_dc = warm_start_sol.get('dc', np.zeros((nD, nI), dtype=bool))
                ws_y = warm_start_sol.get('y_sol', np.zeros((nD, nI, nC)))

                for i in I:
                    am[i].Start = float(ws_am[i])
                    cm[i].Start = float(ws_cm[i])

                for d in D:
                    for i in I:
                        dc[d, i].Start = float(ws_dc[d, i])
                        v[i, d].Start = float(bool(ws_am[i]) and bool(ws_dc[d, i]))
                        w[i, d].Start = float(bool(ws_cm[i]) and bool(ws_dc[d, i]))
                        for c in C:
                            y[d, i, c].Start = float(ws_y[d, i, c] > 0.5)
                    dcAM[d].Start = float(any(bool(ws_am[i]) and bool(ws_dc[d, i]) for i in I))  # [FIX 2]

                # [FIX 3] n3DP ricalcolato sul picco SAA, non sulla media della CG
                for d in D:
                    hours_per_scenario = np.zeros(nS)
                    for i in I:
                        if not (bool(ws_am[i]) and bool(ws_dc[d, i])):
                            continue
                        for s in S:
                            dem_s = sum(d_ics[i, c, s] for c in C if ws_y[d, i, c] > 0.5)
                            hours_per_scenario[s] += a_i[i] * dem_s
                    peak = hours_per_scenario.max() if hours_per_scenario.size else 0.0
                    n3DP[d].Start = float(np.ceil(peak / Cap_3DP)) if peak > 0 else 0.0
                # [FIX 1] B[i,d].Start NON impostato -- lasciato a Gurobi

                mdl.update()
                if verbose:
                    n3dp_dbg = [n3DP[d].Start for d in D]
                    print(f"  [warm start] Vars 1° stadio caricate (binarie/intere, "
                          f"B lasciato a Gurobi, n3DP ricalcolato su picco SAA: {n3dp_dbg}).")
            except Exception as e:
                if verbose: print(f"  [warm start] errore: {e}")

        mdl.optimize()
        cuts_added = None

    else:        # ── Benders multi-cut ─────────────────────────────────────────────────
        # [FIX_BACKORDER_LT] cost_bo_lt e' primo stadio (dipende solo da mu_var,
        # cioe' da y -> deterministico dato il primo stadio): entra qui, non nel
        # subproblema per-scenario. [FIX_RAW_DOUBLE_COUNT] cost_raw tolto.
        mdl.setObjective(
            cost_BS + cost_EOQ + cost_raw_dc + cost_leas + cost_bo_lt
            + gp.quicksum(pi_s * theta[s] for s in S),
            GRB.MINIMIZE)

        mdl._best_ub = float('inf')
        mdl._best_solution = None
        cuts_info = {'count': 0}

        def benders_callback(model, where):
            if where != GRB.Callback.MIPSOL:
                return
            if time.time() - t_start > time_limit * 1.1:
                return

            dc_hat = (np.array([[model.cbGetSolution(dc[d, i]) for i in I] for d in D]) > 0.5).astype(float)
            B_hat = np.maximum(np.array([[model.cbGetSolution(B[i, d]) for d in D] for i in I]), 0.0)
            n3DP_hat = np.array([max(0, round(model.cbGetSolution(n3DP[d]))) for d in D])
            w_hat = (np.array([[model.cbGetSolution(w[i, d]) for d in D] for i in I]) > 0.5).astype(float)
            v_hat = (np.array([[model.cbGetSolution(v[i, d]) for d in D] for i in I]) > 0.5).astype(float)
            am_hat = np.array([model.cbGetSolution(am[i]) > 0.5 for i in I])
            cm_hat = np.array([model.cbGetSolution(cm[i]) > 0.5 for i in I])
            theta_hat = np.array([model.cbGetSolution(theta[s]) for s in S])

            Q_s_list = []
            for s in S:
                sub = gp.Model(f"sub_{s}")
                sub.setParam("OutputFlag", 0)
                sub.setParam("Method", 1)
                sub.setParam("FeasibilityTol", 1e-6)

                q_cm_s = sub.addVars(nI, nD, lb=0.0, name="q_cm")
                q_am_s = sub.addVars(nI, nD, lb=0.0, name="q_am")
                f_s = sub.addVars(nI, nD, nC, lb=0.0, name="f")
                u_s = sub.addVars(nI, nC, lb=0.0, name="u")
                inv_s = sub.addVars(nI, nD, lb=0.0, name="inv")

                sub.addConstrs((gp.quicksum(a_i[i] * q_am_s[i, d] for i in I) <= Cap_3DP * n3DP_hat[d]
                                 for d in D), name="E2")
                sub.addConstrs((B_hat[i, d] + q_cm_s[i, d] + q_am_s[i, d]
                                 - gp.quicksum(f_s[i, d, c] for c in C) == inv_s[i, d]
                                 for i in I for d in D), name="F1")
                sub.addConstrs((gp.quicksum(f_s[i, d, c] for d in D) + u_s[i, c] == d_ics[i, c, s]
                                 for i in I for c in C), name="F2")
                sub.addConstrs((f_s[i, d, c] <= F_bar[i, d] * dc_hat[d, i]
                                 for i in I for d in D for c in C), name="G1")
                sub.addConstrs((q_cm_s[i, d] <= Q_bar_CM[i, d] * w_hat[i, d]
                                 for i in I for d in D), name="G2")
                sub.addConstrs((q_am_s[i, d] <= Q_bar_AM[i, d] * v_hat[i, d]
                                 for i in I for d in D), name="G3")

                cost_T_sub = gp.quicksum(t_dic[d, i, c] * f_s[i, d, c] for i in I for d in D for c in C)
                sub.setObjective(
                    gp.quicksum(c_CM[i] * q_cm_s[i, d] for i in I for d in D) +
                    gp.quicksum(prod_i[i] * q_am_s[i, d] for i in I for d in D) +
                    cost_T_sub +
                    gp.quicksum(h_d[d] * c_CM[i] * inv_s[i, d] for i in I for d in D) +
                    gp.quicksum(b_i[i] * u_s[i, c] for i in I for c in C),
                    GRB.MINIMIZE)
                sub.optimize()

                if sub.Status != GRB.OPTIMAL:
                    continue

                Q_s = sub.ObjVal
                Q_s_list.append(Q_s)

                if theta_hat[s] >= Q_s - 1e-4:
                    continue

                lam_B, lam_dc, lam_w, lam_v, lam_n3DP = {}, {}, {}, {}, {}
                for i in I:
                    for d in D:
                        c_f1 = sub.getConstrByName(f"F1[{i},{d}]")
                        lam_B[i, d] = c_f1.Pi if c_f1 else 0.0
                        c_g2 = sub.getConstrByName(f"G2[{i},{d}]")
                        lam_w[i, d] = c_g2.Pi if c_g2 else 0.0
                        c_g3 = sub.getConstrByName(f"G3[{i},{d}]")
                        lam_v[i, d] = c_g3.Pi if c_g3 else 0.0
                        lam_dc[i, d] = sum(
                            (sub.getConstrByName(f"G1[{i},{d},{c}]").Pi
                             if sub.getConstrByName(f"G1[{i},{d},{c}]") else 0.0)
                            for c in C)
                for d in D:
                    c_e2 = sub.getConstrByName(f"E2[{d}]")
                    lam_n3DP[d] = c_e2.Pi if c_e2 else 0.0

                all_lam = (list(lam_B.values()) + list(lam_dc.values()) +
                           list(lam_w.values()) + list(lam_v.values()) + list(lam_n3DP.values()))
                max_lam = max((abs(v_) for v_ in all_lam), default=1.0)
                if max_lam > 1e-9:
                    scale = max_lam
                    lam_B = {k: v_ / scale for k, v_ in lam_B.items()}
                    lam_dc = {k: v_ / scale for k, v_ in lam_dc.items()}
                    lam_w = {k: v_ / scale for k, v_ in lam_w.items()}
                    lam_v = {k: v_ / scale for k, v_ in lam_v.items()}
                    lam_n3DP = {k: v_ / scale for k, v_ in lam_n3DP.items()}
                    Q_s_norm = Q_s / scale
                else:
                    Q_s_norm = Q_s

                cut = (Q_s_norm
                       + gp.quicksum(lam_B[i, d] * (B[i, d] - B_hat[i, d]) for i in I for d in D)
                       + gp.quicksum(lam_dc[i, d] * (dc[d, i] - dc_hat[d, i]) for i in I for d in D)
                       + gp.quicksum(lam_w[i, d] * (w[i, d] - w_hat[i, d]) for i in I for d in D)
                       + gp.quicksum(lam_v[i, d] * (v[i, d] - v_hat[i, d]) for i in I for d in D)
                       + gp.quicksum(lam_n3DP[d] * (n3DP[d] - n3DP_hat[d]) for d in D))
                model.cbLazy(theta[s] >= cut)
                cuts_info['count'] += 1

            if len(Q_s_list) == nS:
                obj_master = model.cbGet(GRB.Callback.MIPSOL_OBJ)
                first_st_cost = obj_master - sum(pi_s * theta_hat[s] for s in S)
                ub = first_st_cost + sum(pi_s * Q_s for Q_s in Q_s_list)
                if ub < model._best_ub:
                    model._best_ub = ub
                    y_hat = np.array([[[model.cbGetSolution(y[d, i, c]) for c in C] for i in I] for d in D])
                    model._best_solution = (
                        dc_hat.copy(), B_hat.copy(), n3DP_hat.copy(),
                        w_hat.copy(), v_hat.copy(), am_hat.copy(), cm_hat.copy(), y_hat.copy())

        if warm_start_sol is not None:
            try:
                ws_am = warm_start_sol.get('am', np.zeros(nI, dtype=bool))
                ws_cm = warm_start_sol.get('cm', np.ones(nI, dtype=bool))
                ws_dc = warm_start_sol.get('dc', np.zeros((nD, nI), dtype=bool))
                ws_y = warm_start_sol.get('y_sol', np.zeros((nD, nI, nC)))
                for i in I:
                    am[i].Start = float(ws_am[i]); cm[i].Start = float(ws_cm[i])
                for d in D:
                    for i in I:
                        dc[d, i].Start = float(ws_dc[d, i])
                        v[i, d].Start = float(bool(ws_am[i]) and bool(ws_dc[d, i]))
                        w[i, d].Start = float(bool(ws_cm[i]) and bool(ws_dc[d, i]))
                        for c in C:
                            y[d, i, c].Start = float(ws_y[d, i, c] > 0.5)
                    dcAM[d].Start = float(any(bool(ws_am[i]) and bool(ws_dc[d, i]) for i in I))  # [FIX 2]
                # [FIX 3] n3DP sul picco SAA (stesso calcolo del ramo full-space)
                for d in D:
                    hours_per_scenario = np.zeros(nS)
                    for i in I:
                        if not (bool(ws_am[i]) and bool(ws_dc[d, i])):
                            continue
                        for s in S:
                            dem_s = sum(d_ics[i, c, s] for c in C if ws_y[d, i, c] > 0.5)
                            hours_per_scenario[s] += a_i[i] * dem_s
                    peak = hours_per_scenario.max() if hours_per_scenario.size else 0.0
                    n3DP[d].Start = float(np.ceil(peak / Cap_3DP)) if peak > 0 else 0.0
                # [FIX 1] B[i,d].Start NON impostato -- lasciato a Gurobi
                mdl.update()
                if verbose:
                    print("  [warm start] incumbent iniziale caricato (binarie/intere, "
                          "B lasciato a Gurobi, n3DP su picco SAA).")
            except Exception as e:
                if verbose:
                    print(f"  [warm start] ignorato: {e}")

        if fix_am is not None and fix_cm is not None:
            for i in I:
                am[i].lb = am[i].ub = float(bool(fix_am[i]))
                cm[i].lb = cm[i].ub = float(bool(fix_cm[i]))
            if fix_n_dc is not None:
                for i in I:
                    if bool(fix_am[i]) and int(fix_n_dc[i]) > 0:
                        mdl.addConstr(gp.quicksum(dc[d, i] for d in D) <= int(fix_n_dc[i]), name=f"fix_ndc_{i}")
            mdl.update()

        mdl.setParam("LazyConstraints", 1)
        mdl.optimize(benders_callback)
        cuts_added = cuts_info['count']

    solve_time = time.time() - t_start

    # ── 4. POST-PROCESSING ────────────────────────────────────────────────────
    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status': 'INFEASIBLE', 'obj': None, 'obj_real': None, 'gap': None,
                'time': solve_time, 'am': None, 'cm': None, 'dc': None, 'B': None,
                'n3DP': None, 'n_dc_per_sku': None, 'cost_breakdown': None,
                'cuts_added': cuts_added if use_benders else None, 'y_sol': None}

    status = {GRB.OPTIMAL: 'OPTIMAL', GRB.TIME_LIMIT: 'TIME_LIMIT',
              GRB.SUBOPTIMAL: 'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val > 0.5 else 0.0

    if use_benders and hasattr(mdl, '_best_solution') and mdl._best_solution is not None:
        best = mdl._best_solution
        best_dc, best_B, best_n3DP, best_w, best_v, best_am, best_cm, best_y = best
        am_sol, cm_sol = best_am, best_cm
        dc_sol = best_dc > 0.5
        B_sol, n3DP_sol, w_sol, v_sol, y_sol = best_B, best_n3DP, best_w, best_v, best_y
    else:
        am_sol = np.array([am[i].X > 0.5 for i in I])
        cm_sol = np.array([cm[i].X > 0.5 for i in I])
        dc_sol = np.array([[dc[d, i].X > 0.5 for i in I] for d in D])
        B_sol = np.array([[B[i, d].X for d in D] for i in I])
        n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
        w_sol = np.array([[w[i, d].X for d in D] for i in I])
        v_sol = np.array([[v[i, d].X for d in D] for i in I])
        y_sol = np.array([[[snap(y[d, i, c].X) for c in C] for i in I] for d in D])

    n_dc_per_sku = dc_sol.sum(axis=0)

    mu_var_sol = np.array([[sum(mu_ic[i, c] * y_sol[d, i, c] for c in C) for i in I] for d in D])

    C_EOQ_act_sol = np.zeros((nD, nI))
    for d in D:
        for i in I:
            if w_sol[i, d] > 0.5:
                m = int(np.argmin(np.abs(mu_grid - mu_var_sol[d, i])))
                C_EOQ_act_sol[d, i] = C_EOQ_grid[i, d, m]

    mu_raw_sol = np.array([[qbar_i[i] * mu_var_sol[d, i] * float(v_sol[i, d] > 0.5) for i in I] for d in D])

    dcAM_sol = np.array([any(v_sol[i, d] > 0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD)
    B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d, :].sum()
            m = int(np.argmin(np.abs(mu_raw_grid - mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d, m]
            B_raw_sol_arr[d] = B_raw_grid[d, m]

    # [FIX_MCCORMICK] costo base stock reale, tecnologia-specifico
    cost_BS_sol = sum(
        c_CM[i] * (1 + h_d[d]) * B_sol[i, d] * (1 if w_sol[i, d] > 0.5 else 0) +
        prod_i[i] * (1 + h_d[d]) * B_sol[i, d] * (1 if v_sol[i, d] > 0.5 else 0)
        for i in I for d in D)
    cost_EOQ_sol = sum(C_EOQ_act_sol[d, i] for d in D for i in I)
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d] + h_d[d] * c_raw * B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas * n3DP_sol[d] for d in D)
    cost_1st_total = cost_BS_sol + cost_EOQ_sol + cost_raw_dc_sol + cost_leas_sol

    pp = gp.Model("postproc")
    pp.setParam("OutputFlag", 0)
    q_pp = pp.addVars(nI, nD, nS, lb=0.0, name="q_cm")
    qa_pp = pp.addVars(nI, nD, nS, lb=0.0, name="q_am")
    f_pp = pp.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u_pp = pp.addVars(nI, nC, nS, lb=0.0, name="u")
    inv_pp = pp.addVars(nI, nD, nS, lb=0.0, name="inv")

    pp.addConstrs((gp.quicksum(a_i[i] * qa_pp[i, d, s] for i in I) <= Cap_3DP * n3DP_sol[d]
                   for d in D for s in S))
    pp.addConstrs((B_sol[i, d] + q_pp[i, d, s] + qa_pp[i, d, s]
                   - gp.quicksum(f_pp[i, d, c, s] for c in C) == inv_pp[i, d, s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(f_pp[i, d, c, s] for d in D) + u_pp[i, c, s] == d_ics[i, c, s]
                   for i in I for c in C for s in S))
    pp.addConstrs((f_pp[i, d, c, s] <= F_bar[i, d] * float(dc_sol[d, i])
                   for i in I for d in D for c in C for s in S))
    pp.addConstrs((q_pp[i, d, s] <= Q_bar_CM[i, d] * float(w_sol[i, d] > 0.5)
                   for i in I for d in D for s in S))
    pp.addConstrs((qa_pp[i, d, s] <= Q_bar_AM[i, d] * float(v_sol[i, d] > 0.5)
                   for i in I for d in D for s in S))

    pp.setObjective(gp.quicksum(pi_s * (
            gp.quicksum(c_CM[i] * q_pp[i, d, s] for i in I for d in D) +
            gp.quicksum(prod_i[i] * qa_pp[i, d, s] for i in I for d in D) +
            gp.quicksum(t_dic[d, i, c] * f_pp[i, d, c, s] for i in I for d in D for c in C) +
            gp.quicksum(h_d[d] * c_CM[i] * inv_pp[i, d, s] for i in I for d in D) +
            gp.quicksum(b_i[i] * u_pp[i, c, s] for i in I for c in C))
        for s in S), GRB.MINIMIZE)
    pp.optimize()

    # [FIX_BACKORDER_LT] BO_act e' definita nella sezione condivisa PRIMA del
    # branch use_benders, quindi il suo .X e' valido in entrambi i casi (non
    # serve una gestione separata per Benders).
    bo_lt_sol = sum(BO_act[d, i].X for i in I for d in D)

    if pp.Status == GRB.OPTIMAL:
        cost_2nd_real = pp.ObjVal
        C_T_real = sum(pi_s * t_dic[d, i, c] * f_pp[i, d, c, s].X
                       for i in I for d in D for c in C for s in S)
        cost_breakdown = {
            'base_stock': cost_BS_sol,
            'EOQ_CM': cost_EOQ_sol,
            'materia_prima': 0.0,   # [FIX_RAW_DOUBLE_COUNT] non piu' un costo separato
            'EOQ_raw': sum(C_EOQ_raw_sol),
            'holding_raw': sum(h_d[d] * c_raw * B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT': bo_lt_sol,   # [FIX_BACKORDER_LT] nuova voce
            'trasporto': C_T_real,
            'produzione_CM': sum(pi_s * c_CM[i] * q_pp[i, d, s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s * prod_i[i] * qa_pp[i, d, s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s * h_d[d] * c_CM[i] * inv_pp[i, d, s].X for i in I for d in D for s in S),
            'backorder_2nd': sum(pi_s * b_i[i] * u_pp[i, c, s].X for i in I for c in C for s in S),
        }
        fill_rate_sol = {}
        fill_rate_violations = []
        for i in I:
            for c in C:
                dem_att = sum(pi_s * d_ics[i, c, s] for s in S)
                sodd = sum(pi_s * sum(f_pp[i, d, c, s].X for d in D) for s in S)
                fr = sodd / dem_att if dem_att > 1e-9 else 1.0
                fill_rate_sol[i, c] = fr
                if fr < alpha_ic[i, c] - 1e-4:
                    fill_rate_violations.append((i, c, fr, alpha_ic[i, c]))

        obj_real = cost_1st_total + bo_lt_sol + cost_2nd_real
    else:
        cost_breakdown = {}
        obj_real = mdl.ObjVal

    return {
        'status': status, 'obj': mdl.ObjVal, 'obj_real': obj_real, 'gap': mdl.MIPGap * 100,
        'time': solve_time, 'am': am_sol, 'cm': cm_sol, 'dc': dc_sol, 'B': B_sol,
        'n3DP': n3DP_sol, 'n_dc_per_sku': n_dc_per_sku, 'cost_breakdown': cost_breakdown,
        'cuts_added': cuts_added if use_benders else None, 'y_sol': y_sol,
        'fill_rate': fill_rate_sol if pp.Status == GRB.OPTIMAL else {},
        'fill_rate_violations': fill_rate_violations if pp.Status == GRB.OPTIMAL else [],
    }


def print_results_stoc(res, scenario_name, nI, nD, c_CM, prod_i, b_i, L_CM, L_AM):
    print(f"\n{'=' * 65}\nSCENARIO: {scenario_name}\n{'=' * 65}")
    if res['status'] == 'INFEASIBLE':
        print("  INFEASIBLE"); return
    print(f"  Status:          {res['status']}")
    if res.get('obj_real') is not None:
        print(f"  Obiettivo reale: {res['obj_real']:>12.2f} \u20ac")
    print(f"  Obiettivo opt.:  {res['obj']:>12.2f} \u20ac")
    print(f"  MIP gap:         {res['gap']:>11.2f}%")
    print(f"  Tempo:           {res['time']:>11.2f} s")
    if res.get('cuts_added') is not None:
        print(f"  Tagli Benders:   {res['cuts_added']:>11d}")

    print(f"\n  {'SKU':<5} {'Tech':<5} {'DC aperti':<14} {'B medio':<10} "
          f"{'c_CM':>7} {'prod':>7} {'b':>8} {'L_CM[s]':>8} {'L_AM[s]':>8}")
    print(f"  {'-' * 75}")
    for i in range(nI):
        tech = "AM" if res['am'][i] else "CM"
        dc_open = [d for d in range(nD) if res['dc'][d, i]]
        b_mean = res['B'][i, dc_open].mean() if dc_open else 0.0
        l_am = L_AM[i, dc_open[0]] * 52 if dc_open else L_AM[i, 0] * 52
        print(f"  {i:<5} {tech:<5} {str(dc_open):<14} {b_mean:<10.1f} "
              f"{c_CM[i]:>7.0f} {prod_i[i]:>7.0f} {b_i[i]:>8.0f} "
              f"{L_CM[i] * 52:>8.1f} {l_am:>8.1f}")

    print(f"\n  Stampanti AM per DC: {np.round(res['n3DP'], 1)}")

    if res.get('cost_breakdown'):
        totale = 0.0
        print(f"\n  Decomposizione costi:")
        for voce, val in res['cost_breakdown'].items():
            if abs(val) > 0.01:
                print(f"    {voce:<20}: {val:>12.2f} \u20ac")
                totale += val
        print(f"    {'TOTALE reale':<20}: {totale:>12.2f} \u20ac")


# ==============================================================================
# ==============================================================================
# PIPELINE DI ESECUZIONE — CG (warm-start CORRETTO) -> MILP con/senza WS
# ==============================================================================
if __name__ == "__main__":
    cg_params = dict(
        c_CM=c_CM, n_i=n_i, L_CM=L_CM, L_AM=L_AM,
        SL=SL_i, mu_ic=mu_ic, h_d=h_d, b_i=b_i, t_dic=t_dic,
        Leas=Leas, o=50., o_raw=30., c_raw=50., den_raw=1200., unit_raw=0.5
    )

    print("\n" + "=" * 72)
    print("STEP 1 — COLUMN GENERATION (con i tre fix) - 20 SKU misto")
    print("=" * 72)
    res_cg = column_generation(cg_params, nI=nI, nD=nD, nC=nC,
                                Leas=Leas, Cap_3DP=5000., max_iter=80, verbose=True)

    cg_warm_start = {
        'am': res_cg['am'].astype(bool),
        'cm': (1 - res_cg['am']).astype(bool),
        'dc': res_cg['dc'].astype(bool),
        'y_sol': res_cg['y'],
        # 'B' e 'n3DP' non servono piu' in input: B non viene iniettato
        # (lasciato a Gurobi) e n3DP viene ricalcolato internamente sul
        # picco SAA -- vedi [FIX_WARMSTART_INJECTION].
    }

    kwargs_milp = dict(
        nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_ic,
        SL=SL_i, SL_raw=0.85, c_CM=c_CM, o=50., h_d=h_d, b_i=b_i, L_CM=L_CM,
        n_i=n_i, c_raw=50., den_raw=1200., unit_raw=0.5,
        o_raw=30., L_raw=1 / 52, L_AM=L_AM,
        Leas=Leas, Cap_3DP=5000., a_i=a_i,
        t_central=0., t_dic=t_dic, xi=2.0, seed=42,
    )

    # Stesso budget della comparazione "ennesimo confronto" (nS=50,
    # mip_gap=0.05, time_limit=1500), identico in entrambe le run.
    print("\n\n" + "=" * 72)
    print("STEP 2 — MILP FULL-SPACE SAA CON warm-start CG (iniezione corretta)")
    print("=" * 72)
    res_ws = solve_stochastic(
        **kwargs_milp, nS=50, use_benders=False,
        mip_gap=0.05, time_limit=1500,
        warm_start_sol=cg_warm_start, verbose=True,
        scenario_name="FS_20sku_CG_WarmStart_FIXED_INJECTION",
    )
    print_results_stoc(res_ws, "MILP+CG warm-start (iniezione corretta)", nI, nD,
                        c_CM, c_AM, b_i, L_CM, L_AM)

    print("\n\n" + "=" * 72)
    print("STEP 3 — MILP FULL-SPACE SAA SENZA warm-start (confronto)")
    print("=" * 72)
    res_no = solve_stochastic(
        **kwargs_milp, nS=50, use_benders=False,
        mip_gap=0.05, time_limit=1500,
        warm_start_sol=None, verbose=True,
        scenario_name="FS_20sku_no_WarmStart",
    )
    print_results_stoc(res_no, "MILP standard (senza warm-start)", nI, nD,
                        c_CM, c_AM, b_i, L_CM, L_AM)

    print("\n\n" + "=" * 72)
    print("CONFRONTO FINALE")
    print("=" * 72)
    if res_ws['am'] is not None and res_no['am'] is not None:
        am_ws = [i for i in range(nI) if res_ws['am'][i]]
        am_no = [i for i in range(nI) if res_no['am'][i]]
        print(f"  {'Metodo':<25}{'Costo':>14}{'Gap':>10}{'Tempo':>10}{'n_AM':>7}")
        print(f"  {'CG + MILP (WS corretto)':<25}{res_ws['obj_real']:>14,.0f}"
              f"{res_ws['gap']:>9.2f}%{res_ws['time']:>10.1f}{len(am_ws):>7}")
        print(f"  {'MILP standard':<25}{res_no['obj_real']:>14,.0f}"
              f"{res_no['gap']:>9.2f}%{res_no['time']:>10.1f}{len(am_no):>7}")
        print(f"\n  AM (con WS corretto): {am_ws}")
        print(f"  AM (senza WS):        {am_no}")
        print(f"  Coerenza: {sum(1 for i in range(nI) if res_ws['am'][i]==res_no['am'][i])}/{nI} SKU concordi")


STEP 1 — COLUMN GENERATION (con i tre fix) - 20 SKU misto
  [CG] Precomputazione: 7,560 entry in 4s
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2849570
Academic license 2849570 - for non-commercial use only - registered to st___@mail.polimi.it
  [CG] It   0: LB=   224,241  col=  200  new= 80  rc=  23,565  t=4s
  [CG] It   5: LB=   224,241  col=  532  new= 55  rc=  26,230  t=4s
  [CG] It  10: LB=   210,693  col=  581  new=  3  rc=     451  t=4s
  [CG] It  11: LB=   210,690  col=  581  new=  0  rc=       0  t=4s


STEP 2 — MILP FULL-SPACE SAA CON warm-start CG (iniezione corretta)
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 1500
  [varhint] 20/20 SKU con hint AM.
  [warm start] Vars 1° stadio caricate (binarie/intere, B lasciato a Gurobi, n3DP ricalcolato su picco SAA: [1.0, 0.0, 0.0]).
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD EPYC 7

## meno limitata

In [ ]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 20 SKU misto — VERSIONE CON I TRE FIX APPLICATI
================================================================================
Parte dalla versione "Benders Multi-Cut CORRETTO" (cella 2 del notebook
caricato), che gia' includeva i fix FIX_BIGM/FIX_MAP/FIX_INFEAS e la
struttura Base Stock (Acquisto+Immobilizzo). A quella base sono stati
aggiunti i TRE fix mancanti, verificati assenti in ENTRAMBE le celle del
notebook originale (sia la pipeline CG->MILP della cella 1, sia il modello
Benders-capable della cella 2):

  [FIX_MCCORMICK]  Base stock tecnologia-specifico. Prima: cost_BS usava
                    sempre c_CM[i], anche per le SKU AM. Ora: B_cm_lin
                    (gated da w[i,d]) usa c_CM, B_am_lin (gated da v[i,d])
                    usa prod_i — linearizzazione McCormick standard di
                    B[i,d]*w[i,d] e B[i,d]*v[i,d].

  [FIX_BACKORDER_LT] Costo di backorder analitico da lead time, assente
                    prima. bo_cm_aux/bo_am_aux via PWL su mu_var (stessa
                    logica di beta_cm_aux/beta_am_aux gia' presenti),
                    BO_act seleziona il valore giusto per tecnologia via
                    Big-M gated da cm[i]/am[i] e dc[d,i]. Aggiunto a
                    cost_bo_lt nell'obiettivo di PRIMO STADIO, quindi vale
                    automaticamente sia per il full-space sia per Benders
                    (theta[s] resta un costo di secondo stadio puro).

  [FIX_RAW_DOUBLE_COUNT] cost_raw = c_raw*mu_raw_contrib RIMOSSO
                    dall'obiettivo (prima presente, causava doppio
                    conteggio: prod_i = n_i*c_CM e' gia' "tutto incluso"
                    per Knofius et al. — il materiale e' gia' dentro il
                    costo unitario AM). La macchina EOQ_raw/capacita'
                    stampante (mu_raw_contrib, mu_raw_total, n3DP) resta
                    intatta: serve ancora per EOQ scorta polvere e per il
                    vincolo di capacita' di stampa, solo il termine di
                    COSTO diretto del materiale viene tolto.

  [FIX_VARHINT]     L'euristica di branching (VarHintVal/Pri) usava
                    c_CM anche per stimare il risparmio AM. Ora usa
                    prod_i (c_am) per il ramo AM, coerentemente con
                    cost_BS, e include il termine di backorder LT nella
                    stima di breakeven (stessa logica del breakeven a
                    fondo file).

NOTA: le due celle del notebook originale duplicavano lo stesso modello
in due implementazioni parallele (pipeline CG->MILP piu' semplice in
cella 1, versione Benders-capable in cella 2) — e nessuna delle due
aveva i tre fix. Questo file sostituisce ENTRAMBE: la Column Generation
qui sotto e' la stessa di prima ma con gli stessi tre fix applicati
internamente, cosi' il warm-start resta coerente con il MILP finale.
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time

# ==============================================================================
# PARAMETRI GLOBALI - 20 SKU (identici al notebook originale)
# ==============================================================================
nI, nD, nC = 20, 3, 6
seed = 42

coord_DC = np.array([[1.0, 8.0], [1.0, 2.0], [5.0, 5.0]])
coord_C = np.array([[0.5, 9.0], [2.0, 8.0], [3.5, 5.5],
                     [5.5, 4.5], [1.5, 2.5], [0.5, 1.0]])

categorie = (["strutt."] * 2 + ["LT-driver"] * 5 +
             ["borderline"] * 5 + ["meccanico"] * 4 + ["consumab."] * 4)

c_CM = np.array([4000., 3500., 5000., 4000., 3000., 2000., 1500.,
                  1200., 900., 700., 500., 300., 200., 150., 100., 80.,
                  40., 25., 15., 10.])
ratio = np.array([0.38, 0.42, 1.20, 1.30, 1.40, 1.50, 1.60,
                   2.00, 2.50, 3.00, 3.00, 3.50, 3.50, 4.00, 4.50, 5.00,
                   4.00, 5.00, 5.50, 6.00])
c_AM = c_CM * ratio
n_i = ratio

L_CM = np.array([35, 30, 40, 35, 30, 28, 25, 20, 18, 16, 14, 12, 12, 10, 8, 6, 8, 6, 5, 3]) / 52
L_AM_vec = np.array([1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 4]) / 52
L_AM = np.column_stack([L_AM_vec] * nD)

b_i = np.array([100000., 80000., 150000., 120000., 80000., 50000., 40000.,
                15000., 12000., 10000., 8000., 5000., 3000., 2000., 1500., 1000.,
                600., 400., 250., 150.])
SL_i = np.array([0.99, 0.99, 0.99, 0.99, 0.98, 0.97, 0.97,
                  0.95, 0.95, 0.93, 0.92, 0.90, 0.90, 0.90, 0.88, 0.88,
                  0.85, 0.83, 0.82, 0.80])

mu_ic = np.array([
    [0.6, 0.5, 0.4, 0.3, 0.3, 0.3], [0.5, 0.4, 0.4, 0.3, 0.2, 0.2],
    [0.5, 0.4, 0.4, 0.3, 0.4, 0.3], [0.6, 0.4, 0.4, 0.3, 0.4, 0.3],
    [0.8, 0.6, 0.5, 0.4, 0.5, 0.4], [1.0, 0.7, 0.7, 0.6, 0.6, 0.6],
    [1.1, 0.8, 0.7, 0.6, 0.7, 0.7], [1.2, 1.0, 0.9, 0.8, 0.9, 0.8],
    [1.3, 1.1, 1.0, 0.8, 1.0, 0.9], [1.4, 1.1, 1.0, 0.9, 1.0, 0.9],
    [1.0, 0.8, 0.8, 0.7, 0.8, 0.7], [0.9, 0.7, 0.7, 0.6, 0.7, 0.6],
    [2.5, 2.0, 1.8, 1.5, 1.5, 1.2], [2.0, 1.8, 1.5, 1.2, 1.2, 1.0],
    [1.8, 1.5, 1.2, 1.0, 1.0, 0.8], [1.5, 1.2, 1.0, 0.8, 0.8, 0.7],
    [4.0, 3.5, 3.0, 2.5, 2.5, 2.0], [3.5, 3.0, 2.5, 2.0, 2.0, 1.5],
    [3.0, 2.5, 2.0, 1.8, 1.5, 1.5], [2.5, 2.0, 1.8, 1.5, 1.2, 1.2],
])

alpha_ic = np.column_stack([SL_i] * nC)
a_i = c_AM / (1.30 * 0.00525 * 3600)
h_d = np.full(nD, 0.25)
Leas = 15000.

dist = np.array([[np.sqrt((coord_DC[d, 0] - coord_C[c, 0]) ** 2 +
                           (coord_DC[d, 1] - coord_C[c, 1]) ** 2)
                   for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist * 80.0] * nI, axis=1)


# ==============================================================================
# FUNZIONI ANALITICHE (identiche all'originale)
# ==============================================================================
def expected_backorder(mu_L, beta):
    if mu_L < 1e-9 or beta < 0: return mu_L
    if beta == 0: return mu_L
    return max(mu_L * (1 - poisson.cdf(beta - 1, mu_L)) - beta * (1 - poisson.cdf(beta, mu_L)), 0.0)


def expected_on_hand(mu_L, beta):
    return max(beta - mu_L + expected_backorder(mu_L, beta), 0.0)


def optimal_beta(mu_L, c_hold, b_val):
    if mu_L < 1e-9: return 0, 0.0
    best_cost, best_b = float('inf'), 0
    for b in range(0, int(mu_L * 4) + 20):
        tot = c_hold * b + b_val * expected_backorder(mu_L, b)
        if tot < best_cost:
            best_cost, best_b = tot, b
        elif tot > best_cost * 1.5 and b > mu_L:
            break
    return best_b, best_cost


def beta_poisson(mu_val, L_val, SL_target):
    lam = mu_val * L_val
    if lam <= 1e-9: return 0
    for s in range(100000):
        if poisson.cdf(s, lam) >= SL_target: return int(s)
    return 99999


# ==============================================================================
# COLUMN GENERATION — CON I TRE FIX APPLICATI
# ==============================================================================
def precompute_all_columns(nI, nD, nC, p):
    """
    [FIX_MCCORMICK] branch AM: c_hold ora usa c_am (= p['c_CM'][i]*p['n_i'][i]),
                    non piu' p['c_CM'][i].
    [FIX_BACKORDER_LT] aggiunto bo_lt = p['b_i'][i]*expected_backorder(mu_L, beta)
                    a entrambi i branch (CM e AM), sommato al costo colonna.
    [FIX_RAW_DOUBLE_COUNT] rimosso il termine 'raw = c_raw*qraw*mu_id' dal
                    branch AM — prod = c_am*mu_id e' gia' tutto incluso.
    """
    t0 = time.time()
    table = {}
    n_entries = 0
    for i in range(nI):
        table[i] = {}
        for d in range(nD):
            table[i][d] = {}
            for tech in ['CM', 'AM']:
                table[i][d][tech] = {}
                for mask in range(1, 2 ** nC):
                    clients = [c for c in range(nC) if mask & (1 << c)]
                    mu_id = sum(p['mu_ic'][i, c] for c in clients)
                    if mu_id < 1e-9: continue
                    trans = sum(p['t_dic'][d, i, c] * p['mu_ic'][i, c] for c in clients)

                    if tech == 'CM':
                        mu_L = mu_id * p['L_CM'][i]
                        c_hold = p['c_CM'][i] * (1 + p['h_d'][d])
                        beta, bs_back = optimal_beta(mu_L, c_hold, p['b_i'][i])
                        eoq = np.sqrt(2 * p['o'] * p['h_d'][d] * p['c_CM'][i] * mu_id)
                        prod = p['c_CM'][i] * mu_id
                        hold_res = p['h_d'][d] * p['c_CM'][i] * expected_on_hand(mu_L, beta)
                        bo_lt = p['b_i'][i] * expected_backorder(mu_L, beta)   # [FIX_BACKORDER_LT]
                        cost = bs_back + eoq + prod + trans + hold_res + bo_lt
                        ph = 0
                    else:
                        L_am = p['L_AM'][:, 0][i]
                        mu_L = mu_id * L_am
                        c_am = p['c_CM'][i] * p['n_i'][i]
                        c_hold = c_am * (1 + p['h_d'][d])                     # [FIX_MCCORMICK]
                        beta, bs_back = optimal_beta(mu_L, c_hold, p['b_i'][i])
                        prod = c_am * mu_id
                        vol = c_am / (1.30 * 1e6)
                        qraw = vol * p['den_raw'] * p['unit_raw']
                        # [FIX_RAW_DOUBLE_COUNT] rimosso: raw = p['c_raw']*qraw*mu_id
                        eoq_raw = np.sqrt(2 * p['o_raw'] * p['h_d'][d] * p['c_raw'] * max(qraw * mu_id, 1e-9))
                        hold_raw = p['h_d'][d] * p['c_raw'] * qraw * mu_id * 0.5
                        hold_res = p['h_d'][d] * c_am * expected_on_hand(mu_L, beta)  # [FIX_MCCORMICK] c_am non c_CM
                        bo_lt = p['b_i'][i] * expected_backorder(mu_L, beta)   # [FIX_BACKORDER_LT]
                        a_i_local = c_am / (1.30 * 0.00525 * 3600)
                        ph = a_i_local * mu_id
                        cost = bs_back + prod + eoq_raw + hold_raw + trans + hold_res + bo_lt

                    table[i][d][tech][mask] = (cost, mu_id, beta, ph)
                    n_entries += 1

    elapsed = time.time() - t0
    print(f"  [CG] Precomputazione: {n_entries:,} entry in {elapsed:.0f}s")
    return table


def pricing_precomputed(duals_ic, w_vals, table, nI, nD, nC, Leas, existing):
    new_cols = []
    best_rc = 0
    for i in range(nI):
        for d in range(nD):
            for tech in ['CM', 'AM']:
                best_mask_rc, best_mask, best_entry = -1e12, 0, None
                for mask, entry in table[i][d][tech].items():
                    cost, mu, beta, ph = entry
                    dual_sum = sum(duals_ic[i, c] for c in range(nC) if mask & (1 << c))
                    rc = dual_sum - cost
                    if rc > best_mask_rc:
                        best_mask_rc, best_mask, best_entry = rc, mask, entry

                if best_entry is None: continue
                rc_adj = best_mask_rc
                if tech == 'AM' and w_vals[d] < 0.5:
                    rc_adj -= Leas * (1 - w_vals[d])

                if rc_adj > 1e-4:
                    key = (d, i, tech, best_mask)
                    if key not in existing:
                        cost, mu, beta, ph = best_entry
                        new_cols.append((d, i, tech, best_mask, cost, mu, beta, ph, rc_adj))
                        existing.add(key)
                        best_rc = max(best_rc, rc_adj)
    return new_cols, best_rc


def column_generation(params, nI, nD, nC, Leas, Cap_3DP=5000,
                       stock_capacity=None, max_iter=80, verbose=True):
    t0 = time.time()
    table = precompute_all_columns(nI, nD, nC, params)

    columns = []
    full_mask = (1 << nC) - 1
    for i in range(nI):
        for d in range(nD):
            for tech in ['CM', 'AM']:
                if full_mask in table[i][d][tech]:
                    cost, mu, beta, ph = table[i][d][tech][full_mask]
                    columns.append((d, i, tech, full_mask, cost, mu, beta, ph))
    existing = set((c[0], c[1], c[2], c[3]) for c in columns)

    mdl = gp.Model("CG")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    # ── Parametri Aggressivi per chiusura Gap ──
    mdl.setParam("MIPFocus", 2)    # 1=Feasibility, 2=Optimality (Best Bound), 3=Bound stretto
    mdl.setParam("Cuts", 2)        # 0=Off, 1=Moderati, 2=Aggressivi, 3=Molto aggressivi
    mdl.setParam("Presolve", 2)    # 2 = Presolve aggressivo per ridurre la matrice iniziale
    mdl.setParam("Heuristics", 0.05) # Riduciamo le euristiche (default 0.05) dato che abbiamo il Warm Start
    mdl.Params.OutputFlag = 0
    mdl.Params.Method = 1

    n3dp = {}
    for d in range(nD):
        n3dp[d] = mdl.addVar(lb=0, name=f"n3dp_{d}")
    cover = {}
    for i in range(nI):
        for c in range(nC):
            cover[i, c] = mdl.addConstr(gp.LinExpr() == 1, name=f"cv_{i}_{c}")
    cap_c = {}
    for d in range(nD):
        cap_c[d] = mdl.addConstr(-Cap_3DP * n3dp[d] <= 0, name=f"cap_{d}")

    mdl.setObjective(gp.quicksum(Leas * n3dp[d] for d in range(nD)), GRB.MINIMIZE)
    mdl.update()

    lam_vars = []
    for r, col in enumerate(columns):
        dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r = col
        gc = gp.Column()
        for c in range(nC):
            if mask_r & (1 << c): gc.addTerms(1.0, cover[sku_r, c])
        if tech_r == 'AM' and ph_r > 0: gc.addTerms(ph_r, cap_c[dc_r])
        lam_vars.append(mdl.addVar(lb=0, ub=1, obj=cost_r, column=gc, name=f"l_{r}"))
    mdl.update()

    for it in range(max_iter):
        mdl.optimize()
        if mdl.Status != 2: break
        lb = mdl.ObjVal
        duals_ic = np.zeros((nI, nC))
        for i in range(nI):
            for c in range(nC): duals_ic[i, c] = cover[i, c].Pi
        w_vals = np.array([n3dp[d].X for d in range(nD)])

        new_cols, best_rc = pricing_precomputed(duals_ic, w_vals, table, nI, nD, nC, Leas, existing)
        n_new = len(new_cols)

        for nc in new_cols:
            dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r, _ = nc
            columns.append((dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r))
            gc = gp.Column()
            for c in range(nC):
                if mask_r & (1 << c): gc.addTerms(1.0, cover[sku_r, c])
            if tech_r == 'AM' and ph_r > 0: gc.addTerms(ph_r, cap_c[dc_r])
            lam_vars.append(mdl.addVar(lb=0, ub=1, obj=cost_r, column=gc, name=f"l_{len(lam_vars)}"))
        mdl.update()

        if verbose and (it % 5 == 0 or n_new == 0):
            el = time.time() - t0
            print(f"  [CG] It {it:>3}: LB={lb:>10,.0f}  col={len(columns):>5}  "
                  f"new={n_new:>3}  rc={best_rc:>8,.0f}  t={el:.0f}s")
        if n_new == 0: break
    mdl.dispose()

    mdl2 = gp.Model("MILP"); mdl2.Params.OutputFlag = 0; mdl2.Params.TimeLimit = 120
    lam2 = mdl2.addVars(len(columns), vtype=GRB.BINARY, name="l")
    n3dp2 = mdl2.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n")
    for i in range(nI):
        for c in range(nC):
            mdl2.addConstr(gp.quicksum(lam2[r] for r in range(len(columns))
                                        if columns[r][1] == i and (columns[r][3] & (1 << c))) == 1)
    for d in range(nD):
        mdl2.addConstr(gp.quicksum(columns[r][7] * lam2[r] for r in range(len(columns))
                                    if columns[r][0] == d and columns[r][2] == 'AM' and columns[r][7] > 0)
                       <= Cap_3DP * n3dp2[d])
    mdl2.setObjective(gp.quicksum(columns[r][4] * lam2[r] for r in range(len(columns)))
                       + gp.quicksum(Leas * n3dp2[d] for d in range(nD)), GRB.MINIMIZE)
    mdl2.optimize()

    selected, obj_milp, gap, n3dp_sol = [], float('inf'), 100, [0] * nD
    if mdl2.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        selected = [r for r in range(len(columns)) if lam2[r].X > 0.5]
        obj_milp = mdl2.ObjVal; gap = mdl2.MIPGap * 100
        n3dp_sol = [int(round(n3dp2[d].X)) for d in range(nD)]
    mdl2.dispose()

    am = np.zeros(nI, dtype=int); dc_sol = np.zeros((nD, nI), dtype=int)
    y_sol = np.zeros((nD, nI, nC), dtype=int); B_sol = np.zeros((nI, nD))
    for r in selected:
        dc_r, sku_r, tech_r, mask_r, cost_r, mu_r, beta_r, ph_r = columns[r]
        if tech_r == 'AM': am[sku_r] = 1
        dc_sol[dc_r, sku_r] = 1; B_sol[sku_r, dc_r] = beta_r
        for c in range(nC):
            if mask_r & (1 << c): y_sol[dc_r, sku_r, c] = 1

    return {'am': am, 'dc': dc_sol, 'y': y_sol, 'B': B_sol, 'n3dp': n3dp_sol,
            'obj_lp': lb, 'obj_milp': obj_milp, 'gap': gap, 'time': time.time() - t0}


# ==============================================================================
# MODELLO PRIMO STADIO + SECONDO STADIO (full-space o Benders) — CON I FIX
# ==============================================================================
def solve_stochastic(
        nI, nD, nC, nS,
        mu_ic, alpha_ic,
        SL, SL_raw,
        c_CM, o, h_d, b_i, L_CM,
        n_i, c_raw, den_raw, unit_raw,
        o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i,
        t_central,
        B_cap=None,
        t_dic=None,
        costo_km=100.0,
        coord_DC=None,
        coord_C=None,
        L_card=None, U_card=None,
        use_benders=True,
        mip_gap=0.01, time_limit=600,
        xi=2.0, seed=42,
        warm_start_sol=None,
        fix_am=None,
        fix_cm=None,
        fix_n_dc=None,
        verbose=False, scenario_name="default"
):
    """
    RETURNS dict:
      status, obj, obj_real, gap, time,
      am, cm, dc, B, n3DP, n_dc_per_sku,
      y_sol, cost_breakdown, cuts_added
    """
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([
            [np.sqrt((coord_DC[d, 0] - coord_C[c, 0]) ** 2 +
                     (coord_DC[d, 1] - coord_C[c, 1]) ** 2)
             for c in range(nC)] for d in range(nD)])
        t_dc = dist * costo_km
        t_dic = np.stack([t_dc] * nI, axis=1)

    # ── 1. PREPROCESSING ─────────────────────────────────────────────────────
    prod_i = n_i * c_CM
    vol_i = (prod_i / 1.30) * 1e-6
    qbar_i = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    mu_grid_set = {0.0}
    for i in I:
        domande_i = [float(mu_ic[i, c]) for c in C]
        for r in range(1, nC + 1):
            for combo in combinations(domande_i, r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid = np.zeros((nI, nD, M))
    B_AM_grid = np.zeros((nI, nD, M))
    C_EOQ_grid = np.zeros((nI, nD, M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i, d, m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i, d, m] = beta_poisson(mu_m, L_AM[i, d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i, d, m] = np.sqrt(2 * o * h_d[d] * c_CM[i] * mu_m)

    B_bar = B_CM_grid[:, :, -1]
    mu_id_tot = mu_ic.sum(axis=1)

    contributi_raw = [float(qbar_i[i] * mu_id_tot[i]) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI + 1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        max_raw = sum(contributi_raw)
        mu_raw_grid = np.linspace(0, max_raw, 200)
    M_RAW = len(mu_raw_grid)

    B_raw_grid = np.zeros((nD, M_RAW))
    C_EOQ_raw_grid = np.zeros((nD, M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d, m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0:
                C_EOQ_raw_grid[d, m] = np.sqrt(2 * o_raw * h_d[d] * c_raw * mu_m)
        B_bar_raw[d] = B_raw_grid[d, -1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:, :, np.newaxis], size=(nI, nC, nS)).astype(float)

    Q_bar_CM = np.zeros((nI, nD))
    F_bar = np.zeros((nI, nD))
    for i in I:
        for d in D:
            Q_bar_CM[i, d] = d_ics[i, :, :].sum(axis=0).max() * xi
            F_bar[i, d] = d_ics[i, :, :].max() * xi
    Q_bar_AM = np.full((nI, nD), Cap_3DP * xi)

    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── 2. MODELLO GUROBI — PRIMO STADIO ─────────────────────────────────────
    mdl = gp.Model(f"stoc_benders_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    am = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")

    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")

    # [FIX_BACKORDER_LT] nuove variabili ausiliarie
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")

    if not use_benders:
        q_CM = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
        q_AM = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
        f = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
        u = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
        I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")
    else:
        theta = mdl.addVars(nS, lb=0.0, name="theta")

    # ── Vincoli primo stadio (identici all'originale) ────────────────────────
    mdl.addConstrs((am[i] + cm[i] == 1 for i in I), name="A1")
    mdl.addConstrs((y[d, i, c] <= dc[d, i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d, i] <= gp.quicksum(y[d, i, c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d, i, c] for d in D) == 1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d, i] for d in D) >= L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d, i] for d in D) <= U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i, d] <= am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i, d] <= dc[d, i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i, d] >= am[i] + dc[d, i] - 1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i, d] <= cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i, d] <= dc[d, i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i, d] >= cm[i] + dc[d, i] - 1 for i in I for d in D), name="W3")

    mu_max_i = np.array([mu_ic[i, :].sum() for i in I])
    mdl.addConstrs((mu_var[d, i] == gp.quicksum(mu_ic[i, c] * y[d, i, c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d, i] <= mu_max_i[i] * dc[d, i]
                    for d in D for i in I), name="DG2_dc")

    N_PWL = 30
    for i in I:
        mu_pts_i = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL).tolist()
        for d in D:
            beta_cm_pts = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in mu_pts_i]
            mdl.addGenConstrPWL(mu_var[d, i], beta_cm_aux[i, d], mu_pts_i, beta_cm_pts,
                                 name=f"PWL_bCM_{d}_{i}")
            beta_am_pts = [float(beta_poisson(mu, L_AM[i, d], SL_i[i])) for mu in mu_pts_i]
            mdl.addGenConstrPWL(mu_var[d, i], beta_am_aux[i, d], mu_pts_i, beta_am_pts,
                                 name=f"PWL_bAM_{d}_{i}")
            eoq_pts_id = [float(np.sqrt(2 * o * h_d[d] * c_CM[i] * mu)) if mu > 0 else 0.0 for mu in mu_pts_i]
            mdl.addGenConstrPWL(mu_var[d, i], C_EOQ_var[d, i], mu_pts_i, eoq_pts_id,
                                 name=f"PWL_EOQ_{d}_{i}")

            # [FIX_BACKORDER_LT] PWL del costo di backorder analitico da lead time,
            # calcolato punto-per-punto sulla stessa griglia mu_pts_i, usando il
            # beta gia' calcolato per quel punto (beta_cm_pts/beta_am_pts sopra).
            bo_cm_pts = [b_i[i] * expected_backorder(mu_pts_i[k] * L_CM[i], beta_cm_pts[k])
                         for k in range(len(mu_pts_i))]
            mdl.addGenConstrPWL(mu_var[d, i], bo_cm_aux[i, d], mu_pts_i, bo_cm_pts,
                                 name=f"PWL_boCM_{d}_{i}")
            bo_am_pts = [b_i[i] * expected_backorder(mu_pts_i[k] * L_AM[i, d], beta_am_pts[k])
                         for k in range(len(mu_pts_i))]
            mdl.addGenConstrPWL(mu_var[d, i], bo_am_aux[i, d], mu_pts_i, bo_am_pts,
                                 name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i, d] >= beta_cm_aux[i, d] - B_bar[i, d] * (1 - cm[i])
                    for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i, d] >= beta_am_aux[i, d] - B_bar[i, d] * (1 - am[i])
                    for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i, d] <= B_bar[i, d] * dc[d, i]
                    for i in I for d in D), name="DG5")

    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i, d] for i in I) <= B_cap
                        for d in D), name="DG5b")

    # [FIX_BACKORDER_LT] BO_act = valore del ramo tecnologico attivo (Big-M gated
    # da cm[i]/am[i], come gia' si fa per B in DG4a/DG4b), azzerato se dc[d,i]=0.
    M_BO_big = max(max(b_i[i] * mu_max_i[i] for i in I), 1.0) * 2
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d, i] >= bo_cm_aux[i, d] - M_BO_big * (1 - cm[i]), name=f"BOa_{d}_{i}")
            mdl.addConstr(BO_act[d, i] >= bo_am_aux[i, d] - M_BO_big * (1 - am[i]), name=f"BOb_{d}_{i}")
            mdl.addConstr(BO_act[d, i] <= bo_cm_aux[i, d] + M_BO_big * (1 - cm[i]), name=f"BOc_{d}_{i}")
            mdl.addConstr(BO_act[d, i] <= bo_am_aux[i, d] + M_BO_big * (1 - am[i]), name=f"BOd_{d}_{i}")
            mdl.addConstr(BO_act[d, i] <= M_BO_big * dc[d, i], name=f"BOe_{d}_{i}")

    for d in D:
        for i in I:
            mdl.addConstr(dcAM[d] >= v[i, d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d] <= gp.quicksum(v[i, d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i) * sum(mu_ic[i, :].sum() for i in I)) * 2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d, i] <= M_raw_big * v[i, d], name=f"RC1_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d, i] <= qbar_i[i] * mu_var[d, i], name=f"RC2_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d, i] >= qbar_i[i] * mu_var[d, i] - M_raw_big * (1 - v[i, d]),
                          name=f"RC3_{d}_{i}")
        mdl.addConstr(mu_raw_total[d] == gp.quicksum(mu_raw_contrib[d, i] for i in I),
                      name=f"mu_raw_tot_{d}")
        mdl.addConstr(mu_raw_total[d] <= M_raw_big * dcAM[d],
                      name=f"mu_raw_dc_{d}")

    N_PWL_RAW = 30
    mu_raw_max = float(sum(qbar_i[i] * mu_ic[i, :].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max, 0.01), N_PWL_RAW).tolist()
    for d in D:
        b_raw_pts = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        eoq_raw_pts = [float(np.sqrt(2 * o_raw * h_d[d] * c_raw * mu)) if mu > 0 else 0.0 for mu in mu_raw_pts]
        b_raw_aux = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], b_raw_aux, mu_raw_pts, b_raw_pts, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d] >= b_raw_aux, name=f"ER4_{d}")
        mdl.addConstr(B_raw[d] <= B_bar_raw[d] * dcAM[d], name=f"ER5_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, eoq_raw_pts, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max()) * 2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d, i] <= M_EOQ_big * w[i, d], name=f"L_EOQ1_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d, i] <= C_EOQ_var[d, i], name=f"L_EOQ2_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d, i] >= C_EOQ_var[d, i] - M_EOQ_big * (1 - w[i, d]), name=f"L_EOQ3_{d}_{i}")

    ratio_pt = np.where(qbar_i > 1e-9, ptime_i / qbar_i, 0.0)
    mdl.addConstrs((n3DP[d] >= gp.quicksum(ratio_pt[i] * mu_raw_contrib[d, i] for i in I) / Cap_3DP for d in D),
                    name="E1")

    # [FIX_MCCORMICK] B_cm_lin/B_am_lin — linearizzazione McCormick standard
    # di B[i,d]*w[i,d] e B[i,d]*v[i,d]. Sostituisce l'uso diretto di B[i,d]
    # con c_CM in cost_BS, che ignorava la tecnologia.
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i, d])
            mdl.addConstr(B_cm_lin[i, d] <= Bub * w[i, d], name=f"BCL1_{i}_{d}")
            mdl.addConstr(B_cm_lin[i, d] <= B[i, d], name=f"BCL2_{i}_{d}")
            mdl.addConstr(B_cm_lin[i, d] >= B[i, d] - Bub * (1 - w[i, d]), name=f"BCL3_{i}_{d}")
            mdl.addConstr(B_am_lin[i, d] <= Bub * v[i, d], name=f"BAL1_{i}_{d}")
            mdl.addConstr(B_am_lin[i, d] <= B[i, d], name=f"BAL2_{i}_{d}")
            mdl.addConstr(B_am_lin[i, d] >= B[i, d] - Bub * (1 - v[i, d]), name=f"BAL3_{i}_{d}")

    # [FIX_MCCORMICK] cost_BS ora tecnologia-specifico: c_CM per CM, prod_i per AM
    cost_BS = gp.quicksum(
        c_CM[i] * (1 + h_d[d]) * B_cm_lin[i, d] + prod_i[i] * (1 + h_d[d]) * B_am_lin[i, d]
        for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d, i] for i in I for d in D)
    # [FIX_RAW_DOUBLE_COUNT] cost_raw RIMOSSO (prima: c_raw*mu_raw_contrib, doppio
    # conteggio perche' prod_i include gia' il materiale "tutto incluso")
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d] + h_d[d] * c_raw * B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas * n3DP[d] for d in D)
    # [FIX_BACKORDER_LT] nuovo componente di costo di primo stadio
    cost_bo_lt = gp.quicksum(BO_act[d, i] for i in I for d in D)

    # ── 3. SECONDO STADIO: full-space o Benders ──────────────────────────────
    if not use_benders:
        mdl.addConstrs((gp.quicksum(a_i[i] * q_AM[i, d, s] for i in I) <= Cap_3DP * n3DP[d] for d in D for s in S),
                        name="E2")
        mdl.addConstrs((B[i, d] + q_CM[i, d, s] + q_AM[i, d, s] - gp.quicksum(f[i, d, c, s] for c in C) ==
                        I_inv[i, d, s] for i in I for d in D for s in S), name="F1")
        mdl.addConstrs(
            (gp.quicksum(f[i, d, c, s] for d in D) + u[i, c, s] == d_ics[i, c, s] for i in I for c in C for s in S),
            name="F2")
        mdl.addConstrs((f[i, d, c, s] <= F_bar[i, d] * dc[d, i] for i in I for d in D for c in C for s in S),
                        name="G1")
        mdl.addConstrs((q_CM[i, d, s] <= Q_bar_CM[i, d] * w[i, d] for i in I for d in D for s in S), name="G2")
        mdl.addConstrs((q_AM[i, d, s] <= Q_bar_AM[i, d] * v[i, d] for i in I for d in D for s in S), name="G3")
        mdl.addConstrs((gp.quicksum(pi_s * gp.quicksum(f[i, d, c, s] for d in D) for s in S) >=
                        alpha_ic[i, c] * gp.quicksum(pi_s * d_ics[i, c, s] for s in S) for i in I for c in C),
                        name="H")

        cost_2nd = gp.quicksum(pi_s * (
                gp.quicksum(c_CM[i] * q_CM[i, d, s] for i in I for d in D) +
                gp.quicksum(prod_i[i] * q_AM[i, d, s] for i in I for d in D) +
                gp.quicksum(t_dic[d, i, c] * f[i, d, c, s] for i in I for d in D for c in C) +
                gp.quicksum(h_d[d] * c_CM[i] * I_inv[i, d, s] for i in I for d in D) +
                gp.quicksum(b_i[i] * u[i, c, s] for i in I for c in C)
        ) for s in S)

        # [FIX_RAW_DOUBLE_COUNT] cost_raw tolto dalla somma; [FIX_BACKORDER_LT] cost_bo_lt aggiunto
        mdl.setObjective(cost_BS + cost_EOQ + cost_raw_dc + cost_leas + cost_bo_lt + cost_2nd, GRB.MINIMIZE)

        if fix_am is not None and fix_cm is not None:
            for i in I:
                am[i].lb = am[i].ub = float(bool(fix_am[i]))
                cm[i].lb = cm[i].ub = float(bool(fix_cm[i]))
            mdl.update()
            if verbose:
                n_fa = sum(bool(fix_am[i]) for i in I)
                print(f"  [fix_tech] {n_fa}/{nI} SKU fissati AM, {nI - n_fa}/{nI} CM.")

        # [FIX_VARHINT] hint di branching ora tecnologia-specifico: usa prod_i
        # (c_am) invece di c_CM per stimare il risparmio del ramo AM, e include
        # il termine di backorder LT nella stima di netto (coerente con cost_BS
        # e cost_bo_lt sopra, invece della vecchia stima solo-holding).
        for i in I:
            mu_c = float(mu_ic[i, :].sum())
            b_cm_h = beta_poisson(mu_c, L_CM[i], SL_i[i])
            b_am_h = beta_poisson(mu_c, L_AM[i, 0], SL_i[i])
            risp_hold = c_CM[i] * (1 + h_d[0]) * b_cm_h - prod_i[i] * (1 + h_d[0]) * b_am_h
            bo_cm_h = b_i[i] * expected_backorder(mu_c * L_CM[i], b_cm_h)
            bo_am_h = b_i[i] * expected_backorder(mu_c * L_AM[i, 0], b_am_h)
            risp = risp_hold + (bo_cm_h - bo_am_h)
            costo = (prod_i[i] - c_CM[i]) * mu_c
            netto = risp - costo
            Leas_hint = Leas if isinstance(Leas, float) else float(Leas)
            if netto > 0:
                am[i].VarHintVal = 1.0; am[i].VarHintPri = 10; cm[i].VarHintVal = 0.0
            elif -Leas_hint < netto <= 0:
                am[i].VarHintVal = 1.0; am[i].VarHintPri = 5; cm[i].VarHintVal = 0.0
            else:
                am[i].VarHintVal = 0.0; am[i].VarHintPri = 1; cm[i].VarHintVal = 1.0
        mdl.update()
        if verbose:
            n_hint_am = sum(1 for i in I if am[i].VarHintVal > 0.5)
            print(f"  [varhint] {n_hint_am}/{nI} SKU con hint AM.")

        # [FIX_WARMSTART_INJECTION] Iniezione del warm-start CG nel ramo
        # full-space -- corretta secondo i 3 punti gia' descritti in
        # main.md (subsec:meth_cg):
        #   1. NON iniettare B[i,d] -- il beta calcolato analiticamente
        #      dalla CG puo' differire di appena 1 unita' da quello che
        #      il motore PWL di Gurobi produce allo stesso mu_id, e
        #      questo basta a violare F1 e far scartare l'INTERO warm
        #      start. Le variabili continue restano a Gurobi.
        #   2. dcAM[d].Start aggiunto (mancava) -- 1 se almeno una SKU AM
        #      della CG e' assegnata a quel DC nel warm start.
        #   3. n3DP[d].Start ricalcolato sul PICCO di domanda tra gli
        #      scenari SAA (d_ics, gia' generato in fase di
        #      preprocessing con lo stesso seed/nS del modello finale),
        #      non sulla domanda media implicita nel vincolo di
        #      capacita' della CG -- altrimenti il punto iniettato puo'
        #      violare E2 in uno scenario di picco e farsi scartare.
        if warm_start_sol is not None:
            try:
                ws_am = warm_start_sol.get('am', np.zeros(nI, dtype=bool))
                ws_cm = warm_start_sol.get('cm', np.ones(nI, dtype=bool))
                ws_dc = warm_start_sol.get('dc', np.zeros((nD, nI), dtype=bool))
                ws_y = warm_start_sol.get('y_sol', np.zeros((nD, nI, nC)))

                for i in I:
                    am[i].Start = float(ws_am[i])
                    cm[i].Start = float(ws_cm[i])

                for d in D:
                    for i in I:
                        dc[d, i].Start = float(ws_dc[d, i])
                        v[i, d].Start = float(bool(ws_am[i]) and bool(ws_dc[d, i]))
                        w[i, d].Start = float(bool(ws_cm[i]) and bool(ws_dc[d, i]))
                        for c in C:
                            y[d, i, c].Start = float(ws_y[d, i, c] > 0.5)
                    dcAM[d].Start = float(any(bool(ws_am[i]) and bool(ws_dc[d, i]) for i in I))  # [FIX 2]

                # [FIX 3] n3DP ricalcolato sul picco SAA, non sulla media della CG
                for d in D:
                    hours_per_scenario = np.zeros(nS)
                    for i in I:
                        if not (bool(ws_am[i]) and bool(ws_dc[d, i])):
                            continue
                        for s in S:
                            dem_s = sum(d_ics[i, c, s] for c in C if ws_y[d, i, c] > 0.5)
                            hours_per_scenario[s] += a_i[i] * dem_s
                    peak = hours_per_scenario.max() if hours_per_scenario.size else 0.0
                    n3DP[d].Start = float(np.ceil(peak / Cap_3DP)) if peak > 0 else 0.0
                # [FIX 1] B[i,d].Start NON impostato -- lasciato a Gurobi

                mdl.update()
                if verbose:
                    n3dp_dbg = [n3DP[d].Start for d in D]
                    print(f"  [warm start] Vars 1° stadio caricate (binarie/intere, "
                          f"B lasciato a Gurobi, n3DP ricalcolato su picco SAA: {n3dp_dbg}).")
            except Exception as e:
                if verbose: print(f"  [warm start] errore: {e}")

        mdl.optimize()
        cuts_added = None

    else:        # ── Benders multi-cut ─────────────────────────────────────────────────
        # [FIX_BACKORDER_LT] cost_bo_lt e' primo stadio (dipende solo da mu_var,
        # cioe' da y -> deterministico dato il primo stadio): entra qui, non nel
        # subproblema per-scenario. [FIX_RAW_DOUBLE_COUNT] cost_raw tolto.
        mdl.setObjective(
            cost_BS + cost_EOQ + cost_raw_dc + cost_leas + cost_bo_lt
            + gp.quicksum(pi_s * theta[s] for s in S),
            GRB.MINIMIZE)

        mdl._best_ub = float('inf')
        mdl._best_solution = None
        cuts_info = {'count': 0}

        def benders_callback(model, where):
            if where != GRB.Callback.MIPSOL:
                return
            if time.time() - t_start > time_limit * 1.1:
                return

            dc_hat = (np.array([[model.cbGetSolution(dc[d, i]) for i in I] for d in D]) > 0.5).astype(float)
            B_hat = np.maximum(np.array([[model.cbGetSolution(B[i, d]) for d in D] for i in I]), 0.0)
            n3DP_hat = np.array([max(0, round(model.cbGetSolution(n3DP[d]))) for d in D])
            w_hat = (np.array([[model.cbGetSolution(w[i, d]) for d in D] for i in I]) > 0.5).astype(float)
            v_hat = (np.array([[model.cbGetSolution(v[i, d]) for d in D] for i in I]) > 0.5).astype(float)
            am_hat = np.array([model.cbGetSolution(am[i]) > 0.5 for i in I])
            cm_hat = np.array([model.cbGetSolution(cm[i]) > 0.5 for i in I])
            theta_hat = np.array([model.cbGetSolution(theta[s]) for s in S])

            Q_s_list = []
            for s in S:
                sub = gp.Model(f"sub_{s}")
                sub.setParam("OutputFlag", 0)
                sub.setParam("Method", 1)
                sub.setParam("FeasibilityTol", 1e-6)

                q_cm_s = sub.addVars(nI, nD, lb=0.0, name="q_cm")
                q_am_s = sub.addVars(nI, nD, lb=0.0, name="q_am")
                f_s = sub.addVars(nI, nD, nC, lb=0.0, name="f")
                u_s = sub.addVars(nI, nC, lb=0.0, name="u")
                inv_s = sub.addVars(nI, nD, lb=0.0, name="inv")

                sub.addConstrs((gp.quicksum(a_i[i] * q_am_s[i, d] for i in I) <= Cap_3DP * n3DP_hat[d]
                                 for d in D), name="E2")
                sub.addConstrs((B_hat[i, d] + q_cm_s[i, d] + q_am_s[i, d]
                                 - gp.quicksum(f_s[i, d, c] for c in C) == inv_s[i, d]
                                 for i in I for d in D), name="F1")
                sub.addConstrs((gp.quicksum(f_s[i, d, c] for d in D) + u_s[i, c] == d_ics[i, c, s]
                                 for i in I for c in C), name="F2")
                sub.addConstrs((f_s[i, d, c] <= F_bar[i, d] * dc_hat[d, i]
                                 for i in I for d in D for c in C), name="G1")
                sub.addConstrs((q_cm_s[i, d] <= Q_bar_CM[i, d] * w_hat[i, d]
                                 for i in I for d in D), name="G2")
                sub.addConstrs((q_am_s[i, d] <= Q_bar_AM[i, d] * v_hat[i, d]
                                 for i in I for d in D), name="G3")

                cost_T_sub = gp.quicksum(t_dic[d, i, c] * f_s[i, d, c] for i in I for d in D for c in C)
                sub.setObjective(
                    gp.quicksum(c_CM[i] * q_cm_s[i, d] for i in I for d in D) +
                    gp.quicksum(prod_i[i] * q_am_s[i, d] for i in I for d in D) +
                    cost_T_sub +
                    gp.quicksum(h_d[d] * c_CM[i] * inv_s[i, d] for i in I for d in D) +
                    gp.quicksum(b_i[i] * u_s[i, c] for i in I for c in C),
                    GRB.MINIMIZE)
                sub.optimize()

                if sub.Status != GRB.OPTIMAL:
                    continue

                Q_s = sub.ObjVal
                Q_s_list.append(Q_s)

                if theta_hat[s] >= Q_s - 1e-4:
                    continue

                lam_B, lam_dc, lam_w, lam_v, lam_n3DP = {}, {}, {}, {}, {}
                for i in I:
                    for d in D:
                        c_f1 = sub.getConstrByName(f"F1[{i},{d}]")
                        lam_B[i, d] = c_f1.Pi if c_f1 else 0.0
                        c_g2 = sub.getConstrByName(f"G2[{i},{d}]")
                        lam_w[i, d] = c_g2.Pi if c_g2 else 0.0
                        c_g3 = sub.getConstrByName(f"G3[{i},{d}]")
                        lam_v[i, d] = c_g3.Pi if c_g3 else 0.0
                        lam_dc[i, d] = sum(
                            (sub.getConstrByName(f"G1[{i},{d},{c}]").Pi
                             if sub.getConstrByName(f"G1[{i},{d},{c}]") else 0.0)
                            for c in C)
                for d in D:
                    c_e2 = sub.getConstrByName(f"E2[{d}]")
                    lam_n3DP[d] = c_e2.Pi if c_e2 else 0.0

                all_lam = (list(lam_B.values()) + list(lam_dc.values()) +
                           list(lam_w.values()) + list(lam_v.values()) + list(lam_n3DP.values()))
                max_lam = max((abs(v_) for v_ in all_lam), default=1.0)
                if max_lam > 1e-9:
                    scale = max_lam
                    lam_B = {k: v_ / scale for k, v_ in lam_B.items()}
                    lam_dc = {k: v_ / scale for k, v_ in lam_dc.items()}
                    lam_w = {k: v_ / scale for k, v_ in lam_w.items()}
                    lam_v = {k: v_ / scale for k, v_ in lam_v.items()}
                    lam_n3DP = {k: v_ / scale for k, v_ in lam_n3DP.items()}
                    Q_s_norm = Q_s / scale
                else:
                    Q_s_norm = Q_s

                cut = (Q_s_norm
                       + gp.quicksum(lam_B[i, d] * (B[i, d] - B_hat[i, d]) for i in I for d in D)
                       + gp.quicksum(lam_dc[i, d] * (dc[d, i] - dc_hat[d, i]) for i in I for d in D)
                       + gp.quicksum(lam_w[i, d] * (w[i, d] - w_hat[i, d]) for i in I for d in D)
                       + gp.quicksum(lam_v[i, d] * (v[i, d] - v_hat[i, d]) for i in I for d in D)
                       + gp.quicksum(lam_n3DP[d] * (n3DP[d] - n3DP_hat[d]) for d in D))
                model.cbLazy(theta[s] >= cut)
                cuts_info['count'] += 1

            if len(Q_s_list) == nS:
                obj_master = model.cbGet(GRB.Callback.MIPSOL_OBJ)
                first_st_cost = obj_master - sum(pi_s * theta_hat[s] for s in S)
                ub = first_st_cost + sum(pi_s * Q_s for Q_s in Q_s_list)
                if ub < model._best_ub:
                    model._best_ub = ub
                    y_hat = np.array([[[model.cbGetSolution(y[d, i, c]) for c in C] for i in I] for d in D])
                    model._best_solution = (
                        dc_hat.copy(), B_hat.copy(), n3DP_hat.copy(),
                        w_hat.copy(), v_hat.copy(), am_hat.copy(), cm_hat.copy(), y_hat.copy())

        if warm_start_sol is not None:
            try:
                ws_am = warm_start_sol.get('am', np.zeros(nI, dtype=bool))
                ws_cm = warm_start_sol.get('cm', np.ones(nI, dtype=bool))
                ws_dc = warm_start_sol.get('dc', np.zeros((nD, nI), dtype=bool))
                ws_y = warm_start_sol.get('y_sol', np.zeros((nD, nI, nC)))
                for i in I:
                    am[i].Start = float(ws_am[i]); cm[i].Start = float(ws_cm[i])
                for d in D:
                    for i in I:
                        dc[d, i].Start = float(ws_dc[d, i])
                        v[i, d].Start = float(bool(ws_am[i]) and bool(ws_dc[d, i]))
                        w[i, d].Start = float(bool(ws_cm[i]) and bool(ws_dc[d, i]))
                        for c in C:
                            y[d, i, c].Start = float(ws_y[d, i, c] > 0.5)
                    dcAM[d].Start = float(any(bool(ws_am[i]) and bool(ws_dc[d, i]) for i in I))  # [FIX 2]
                # [FIX 3] n3DP sul picco SAA (stesso calcolo del ramo full-space)
                for d in D:
                    hours_per_scenario = np.zeros(nS)
                    for i in I:
                        if not (bool(ws_am[i]) and bool(ws_dc[d, i])):
                            continue
                        for s in S:
                            dem_s = sum(d_ics[i, c, s] for c in C if ws_y[d, i, c] > 0.5)
                            hours_per_scenario[s] += a_i[i] * dem_s
                    peak = hours_per_scenario.max() if hours_per_scenario.size else 0.0
                    n3DP[d].Start = float(np.ceil(peak / Cap_3DP)) if peak > 0 else 0.0
                # [FIX 1] B[i,d].Start NON impostato -- lasciato a Gurobi
                mdl.update()
                if verbose:
                    print("  [warm start] incumbent iniziale caricato (binarie/intere, "
                          "B lasciato a Gurobi, n3DP su picco SAA).")
            except Exception as e:
                if verbose:
                    print(f"  [warm start] ignorato: {e}")

        if fix_am is not None and fix_cm is not None:
            for i in I:
                am[i].lb = am[i].ub = float(bool(fix_am[i]))
                cm[i].lb = cm[i].ub = float(bool(fix_cm[i]))
            if fix_n_dc is not None:
                for i in I:
                    if bool(fix_am[i]) and int(fix_n_dc[i]) > 0:
                        mdl.addConstr(gp.quicksum(dc[d, i] for d in D) <= int(fix_n_dc[i]), name=f"fix_ndc_{i}")
            mdl.update()

        mdl.setParam("LazyConstraints", 1)
        mdl.optimize(benders_callback)
        cuts_added = cuts_info['count']

    solve_time = time.time() - t_start

    # ── 4. POST-PROCESSING ────────────────────────────────────────────────────
    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status': 'INFEASIBLE', 'obj': None, 'obj_real': None, 'gap': None,
                'time': solve_time, 'am': None, 'cm': None, 'dc': None, 'B': None,
                'n3DP': None, 'n_dc_per_sku': None, 'cost_breakdown': None,
                'cuts_added': cuts_added if use_benders else None, 'y_sol': None}

    status = {GRB.OPTIMAL: 'OPTIMAL', GRB.TIME_LIMIT: 'TIME_LIMIT',
              GRB.SUBOPTIMAL: 'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val > 0.5 else 0.0

    if use_benders and hasattr(mdl, '_best_solution') and mdl._best_solution is not None:
        best = mdl._best_solution
        best_dc, best_B, best_n3DP, best_w, best_v, best_am, best_cm, best_y = best
        am_sol, cm_sol = best_am, best_cm
        dc_sol = best_dc > 0.5
        B_sol, n3DP_sol, w_sol, v_sol, y_sol = best_B, best_n3DP, best_w, best_v, best_y
    else:
        am_sol = np.array([am[i].X > 0.5 for i in I])
        cm_sol = np.array([cm[i].X > 0.5 for i in I])
        dc_sol = np.array([[dc[d, i].X > 0.5 for i in I] for d in D])
        B_sol = np.array([[B[i, d].X for d in D] for i in I])
        n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
        w_sol = np.array([[w[i, d].X for d in D] for i in I])
        v_sol = np.array([[v[i, d].X for d in D] for i in I])
        y_sol = np.array([[[snap(y[d, i, c].X) for c in C] for i in I] for d in D])

    n_dc_per_sku = dc_sol.sum(axis=0)

    mu_var_sol = np.array([[sum(mu_ic[i, c] * y_sol[d, i, c] for c in C) for i in I] for d in D])

    C_EOQ_act_sol = np.zeros((nD, nI))
    for d in D:
        for i in I:
            if w_sol[i, d] > 0.5:
                m = int(np.argmin(np.abs(mu_grid - mu_var_sol[d, i])))
                C_EOQ_act_sol[d, i] = C_EOQ_grid[i, d, m]

    mu_raw_sol = np.array([[qbar_i[i] * mu_var_sol[d, i] * float(v_sol[i, d] > 0.5) for i in I] for d in D])

    dcAM_sol = np.array([any(v_sol[i, d] > 0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD)
    B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d, :].sum()
            m = int(np.argmin(np.abs(mu_raw_grid - mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d, m]
            B_raw_sol_arr[d] = B_raw_grid[d, m]

    # [FIX_MCCORMICK] costo base stock reale, tecnologia-specifico
    cost_BS_sol = sum(
        c_CM[i] * (1 + h_d[d]) * B_sol[i, d] * (1 if w_sol[i, d] > 0.5 else 0) +
        prod_i[i] * (1 + h_d[d]) * B_sol[i, d] * (1 if v_sol[i, d] > 0.5 else 0)
        for i in I for d in D)
    cost_EOQ_sol = sum(C_EOQ_act_sol[d, i] for d in D for i in I)
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d] + h_d[d] * c_raw * B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas * n3DP_sol[d] for d in D)
    cost_1st_total = cost_BS_sol + cost_EOQ_sol + cost_raw_dc_sol + cost_leas_sol

    pp = gp.Model("postproc")
    pp.setParam("OutputFlag", 0)
    q_pp = pp.addVars(nI, nD, nS, lb=0.0, name="q_cm")
    qa_pp = pp.addVars(nI, nD, nS, lb=0.0, name="q_am")
    f_pp = pp.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u_pp = pp.addVars(nI, nC, nS, lb=0.0, name="u")
    inv_pp = pp.addVars(nI, nD, nS, lb=0.0, name="inv")

    pp.addConstrs((gp.quicksum(a_i[i] * qa_pp[i, d, s] for i in I) <= Cap_3DP * n3DP_sol[d]
                   for d in D for s in S))
    pp.addConstrs((B_sol[i, d] + q_pp[i, d, s] + qa_pp[i, d, s]
                   - gp.quicksum(f_pp[i, d, c, s] for c in C) == inv_pp[i, d, s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(f_pp[i, d, c, s] for d in D) + u_pp[i, c, s] == d_ics[i, c, s]
                   for i in I for c in C for s in S))
    pp.addConstrs((f_pp[i, d, c, s] <= F_bar[i, d] * float(dc_sol[d, i])
                   for i in I for d in D for c in C for s in S))
    pp.addConstrs((q_pp[i, d, s] <= Q_bar_CM[i, d] * float(w_sol[i, d] > 0.5)
                   for i in I for d in D for s in S))
    pp.addConstrs((qa_pp[i, d, s] <= Q_bar_AM[i, d] * float(v_sol[i, d] > 0.5)
                   for i in I for d in D for s in S))

    pp.setObjective(gp.quicksum(pi_s * (
            gp.quicksum(c_CM[i] * q_pp[i, d, s] for i in I for d in D) +
            gp.quicksum(prod_i[i] * qa_pp[i, d, s] for i in I for d in D) +
            gp.quicksum(t_dic[d, i, c] * f_pp[i, d, c, s] for i in I for d in D for c in C) +
            gp.quicksum(h_d[d] * c_CM[i] * inv_pp[i, d, s] for i in I for d in D) +
            gp.quicksum(b_i[i] * u_pp[i, c, s] for i in I for c in C))
        for s in S), GRB.MINIMIZE)
    pp.optimize()

    # [FIX_BACKORDER_LT] BO_act e' definita nella sezione condivisa PRIMA del
    # branch use_benders, quindi il suo .X e' valido in entrambi i casi (non
    # serve una gestione separata per Benders).
    bo_lt_sol = sum(BO_act[d, i].X for i in I for d in D)

    if pp.Status == GRB.OPTIMAL:
        cost_2nd_real = pp.ObjVal
        C_T_real = sum(pi_s * t_dic[d, i, c] * f_pp[i, d, c, s].X
                       for i in I for d in D for c in C for s in S)
        cost_breakdown = {
            'base_stock': cost_BS_sol,
            'EOQ_CM': cost_EOQ_sol,
            'materia_prima': 0.0,   # [FIX_RAW_DOUBLE_COUNT] non piu' un costo separato
            'EOQ_raw': sum(C_EOQ_raw_sol),
            'holding_raw': sum(h_d[d] * c_raw * B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT': bo_lt_sol,   # [FIX_BACKORDER_LT] nuova voce
            'trasporto': C_T_real,
            'produzione_CM': sum(pi_s * c_CM[i] * q_pp[i, d, s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s * prod_i[i] * qa_pp[i, d, s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s * h_d[d] * c_CM[i] * inv_pp[i, d, s].X for i in I for d in D for s in S),
            'backorder_2nd': sum(pi_s * b_i[i] * u_pp[i, c, s].X for i in I for c in C for s in S),
        }
        fill_rate_sol = {}
        fill_rate_violations = []
        for i in I:
            for c in C:
                dem_att = sum(pi_s * d_ics[i, c, s] for s in S)
                sodd = sum(pi_s * sum(f_pp[i, d, c, s].X for d in D) for s in S)
                fr = sodd / dem_att if dem_att > 1e-9 else 1.0
                fill_rate_sol[i, c] = fr
                if fr < alpha_ic[i, c] - 1e-4:
                    fill_rate_violations.append((i, c, fr, alpha_ic[i, c]))

        obj_real = cost_1st_total + bo_lt_sol + cost_2nd_real
    else:
        cost_breakdown = {}
        obj_real = mdl.ObjVal

    return {
        'status': status, 'obj': mdl.ObjVal, 'obj_real': obj_real, 'gap': mdl.MIPGap * 100,
        'time': solve_time, 'am': am_sol, 'cm': cm_sol, 'dc': dc_sol, 'B': B_sol,
        'n3DP': n3DP_sol, 'n_dc_per_sku': n_dc_per_sku, 'cost_breakdown': cost_breakdown,
        'cuts_added': cuts_added if use_benders else None, 'y_sol': y_sol,
        'fill_rate': fill_rate_sol if pp.Status == GRB.OPTIMAL else {},
        'fill_rate_violations': fill_rate_violations if pp.Status == GRB.OPTIMAL else [],
    }


def print_results_stoc(res, scenario_name, nI, nD, c_CM, prod_i, b_i, L_CM, L_AM):
    print(f"\n{'=' * 65}\nSCENARIO: {scenario_name}\n{'=' * 65}")
    if res['status'] == 'INFEASIBLE':
        print("  INFEASIBLE"); return
    print(f"  Status:          {res['status']}")
    if res.get('obj_real') is not None:
        print(f"  Obiettivo reale: {res['obj_real']:>12.2f} \u20ac")
    print(f"  Obiettivo opt.:  {res['obj']:>12.2f} \u20ac")
    print(f"  MIP gap:         {res['gap']:>11.2f}%")
    print(f"  Tempo:           {res['time']:>11.2f} s")
    if res.get('cuts_added') is not None:
        print(f"  Tagli Benders:   {res['cuts_added']:>11d}")

    print(f"\n  {'SKU':<5} {'Tech':<5} {'DC aperti':<14} {'B medio':<10} "
          f"{'c_CM':>7} {'prod':>7} {'b':>8} {'L_CM[s]':>8} {'L_AM[s]':>8}")
    print(f"  {'-' * 75}")
    for i in range(nI):
        tech = "AM" if res['am'][i] else "CM"
        dc_open = [d for d in range(nD) if res['dc'][d, i]]
        b_mean = res['B'][i, dc_open].mean() if dc_open else 0.0
        l_am = L_AM[i, dc_open[0]] * 52 if dc_open else L_AM[i, 0] * 52
        print(f"  {i:<5} {tech:<5} {str(dc_open):<14} {b_mean:<10.1f} "
              f"{c_CM[i]:>7.0f} {prod_i[i]:>7.0f} {b_i[i]:>8.0f} "
              f"{L_CM[i] * 52:>8.1f} {l_am:>8.1f}")

    print(f"\n  Stampanti AM per DC: {np.round(res['n3DP'], 1)}")

    if res.get('cost_breakdown'):
        totale = 0.0
        print(f"\n  Decomposizione costi:")
        for voce, val in res['cost_breakdown'].items():
            if abs(val) > 0.01:
                print(f"    {voce:<20}: {val:>12.2f} \u20ac")
                totale += val
        print(f"    {'TOTALE reale':<20}: {totale:>12.2f} \u20ac")


# ==============================================================================
# ==============================================================================
# PIPELINE DI ESECUZIONE — CG (warm-start CORRETTO) -> MILP con/senza WS
# ==============================================================================
if __name__ == "__main__":
    cg_params = dict(
        c_CM=c_CM, n_i=n_i, L_CM=L_CM, L_AM=L_AM,
        SL=SL_i, mu_ic=mu_ic, h_d=h_d, b_i=b_i, t_dic=t_dic,
        Leas=Leas, o=50., o_raw=30., c_raw=50., den_raw=1200., unit_raw=0.5
    )

    print("\n" + "=" * 72)
    print("STEP 1 — COLUMN GENERATION (con i tre fix) - 20 SKU misto")
    print("=" * 72)
    res_cg = column_generation(cg_params, nI=nI, nD=nD, nC=nC,
                                Leas=Leas, Cap_3DP=5000., max_iter=80, verbose=True)

    cg_warm_start = {
        'am': res_cg['am'].astype(bool),
        'cm': (1 - res_cg['am']).astype(bool),
        'dc': res_cg['dc'].astype(bool),
        'y_sol': res_cg['y'],
        # 'B' e 'n3DP' non servono piu' in input: B non viene iniettato
        # (lasciato a Gurobi) e n3DP viene ricalcolato internamente sul
        # picco SAA -- vedi [FIX_WARMSTART_INJECTION].
    }

    kwargs_milp = dict(
        nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_ic,
        SL=SL_i, SL_raw=0.85, c_CM=c_CM, o=50., h_d=h_d, b_i=b_i, L_CM=L_CM,
        n_i=n_i, c_raw=50., den_raw=1200., unit_raw=0.5,
        o_raw=30., L_raw=1 / 52, L_AM=L_AM,
        Leas=Leas, Cap_3DP=5000., a_i=a_i,
        t_central=0., t_dic=t_dic, xi=2.0, seed=42,
    )

    # Stesso budget della comparazione "ennesimo confronto" (nS=50,
    # mip_gap=0.05, time_limit=1500), identico in entrambe le run.
    print("\n\n" + "=" * 72)
    print("STEP 2 — MILP FULL-SPACE SAA CON warm-start CG (iniezione corretta)")
    print("=" * 72)
    res_ws = solve_stochastic(
        **kwargs_milp, nS=30, use_benders=False,
        mip_gap=0.02, time_limit=14400,
        warm_start_sol=cg_warm_start, verbose=True,
        scenario_name="FS_20sku_CG_WarmStart_FIXED_INJECTION",
    )
    print_results_stoc(res_ws, "MILP+CG warm-start (iniezione corretta)", nI, nD,
                        c_CM, c_AM, b_i, L_CM, L_AM)

    print("\n\n" + "=" * 72)
    print("STEP 3 — MILP FULL-SPACE SAA SENZA warm-start (confronto)")
    print("=" * 72)
    res_no = solve_stochastic(
        **kwargs_milp, nS=30, use_benders=False,
        mip_gap=0.02, time_limit=14400,
        warm_start_sol=None, verbose=True,
        scenario_name="FS_20sku_no_WarmStart",
    )
    print_results_stoc(res_no, "MILP standard (senza warm-start)", nI, nD,
                        c_CM, c_AM, b_i, L_CM, L_AM)

    print("\n\n" + "=" * 72)
    print("CONFRONTO FINALE")
    print("=" * 72)
    if res_ws['am'] is not None and res_no['am'] is not None:
        am_ws = [i for i in range(nI) if res_ws['am'][i]]
        am_no = [i for i in range(nI) if res_no['am'][i]]
        print(f"  {'Metodo':<25}{'Costo':>14}{'Gap':>10}{'Tempo':>10}{'n_AM':>7}")
        print(f"  {'CG + MILP (WS corretto)':<25}{res_ws['obj_real']:>14,.0f}"
              f"{res_ws['gap']:>9.2f}%{res_ws['time']:>10.1f}{len(am_ws):>7}")
        print(f"  {'MILP standard':<25}{res_no['obj_real']:>14,.0f}"
              f"{res_no['gap']:>9.2f}%{res_no['time']:>10.1f}{len(am_no):>7}")
        print(f"\n  AM (con WS corretto): {am_ws}")
        print(f"  AM (senza WS):        {am_no}")
        print(f"  Coerenza: {sum(1 for i in range(nI) if res_ws['am'][i]==res_no['am'][i])}/{nI} SKU concordi")